# Fish2 Raphe PLOTS — selected raphe figure-ready notebook, cleaned + selected-raphe reclustering sweep

This plotting-only notebook is for the **selected ~234 raphe neurons only**. It reads the previous v14 clustering outputs and focuses on:

1. checking cluster membership for selected cell IDs;
2. plotting where those IDs fall on the existing selected-raphe UMAP/MDS;
3. making clean figure-ready UMAP/MDS plots;
4. making fixed-camera 3D cluster views using the same nice Plotly brain shell as the interactive viewer;
5. exporting selected clusters as HTML/PDF/PNG/SVG.

It does **not** rerun fishFuncEM, NBLAST, QC, or clustering.


**v22 addition:** after loading the previous selected-raphe NBLAST outputs, this notebook can re-cut the existing NBLAST distance matrix into multiple cluster numbers (`k`) without recomputing NBLAST. Choose `SELECTED_RAPHE_ACTIVE_K` and all downstream cluster checking / figure-ready 3D export cells use that active reclustering.


**Update:** brain shell rendering now keeps only the largest connected shell component to remove detached ROI/mask blobs, and uses a lower transparency for a cleaner figure-ready outline.

In [ ]:
# ============================================================
# 1. Imports + configuration
# ============================================================

from pathlib import Path
from collections import defaultdict
import os
import re
import json
import math
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
from tqdm import tqdm

# Plotly for interactive cluster rendering
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except Exception as e:
    HAS_PLOTLY = False
    print("Plotly not available:", repr(e))

# Optional widgets for cluster selector
try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except Exception as e:
    HAS_WIDGETS = False
    print("ipywidgets not available:", repr(e))

# Optional neuPrint soma metadata fetching.
try:
    from neuprint import Client, fetch_neurons, NeuronCriteria as NC
    HAS_NEUPRINT = True
except Exception as e:
    HAS_NEUPRINT = False
    print("neuprint-python not available:", repr(e))

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
SKELETON_DIR = PROJECT_DIR / "skeletons"

# This should be the output folder from your previous working v14 notebook.
PREVIOUS_OUT_ROOT = PROJECT_DIR / "fish2_raphe_morphology_v14_selected_vs_raphe_superior_outputs"

# This notebook writes only new/updated plots and tables here.
PLOT_OUT_ROOT = PREVIOUS_OUT_ROOT / "plotting_selected_raphe_clean_figure_ready_v22_reclusterK"
PLOT_OUT_ROOT.mkdir(parents=True, exist_ok=True)

SELECTED_RAW_SWC_DIR = SKELETON_DIR / "selected_raw_swc"
RAPHE_SUPERIOR_RAW_SWC_DIR = SKELETON_DIR / "raphe_superior_raw_swc"

# Dataset keys from v14.
DATASET_CONFIG = {
    "selected_raphe": {
        "label": "Selected raphe neurons",
        "raw_swc_dir": SELECTED_RAW_SWC_DIR,
    },
}

# Figure export settings.
FIG_DPI = 300
SAVE_PNG = True
SAVE_PDF = True
SAVE_SVG = True
SAVE_INTERACTIVE_HTML = True

# Rendering options.
SHOW_BRAIN_SHELL = True
SHELL_ALPHA = 0.055
CELL_LINE_WIDTH = 2.4  # Tune this: smaller values make crowded clusters easier to inspect
CONTEXT_LINE_WIDTH = 0.7
POPULATION_CONTEXT_ALPHA = 0.035
MAX_CONTEXT_CELLS = 250
MAX_CELLS_PER_CLUSTER_RENDER = 60

# v18 cluster-view modes.
# "all_three" displays three separate brains: skeletons-only, somas-only, and combined.
# Other options display one brain.
DEFAULT_CLUSTER_RENDER_MODE = "all_three"
CLUSTER_RENDER_MODE_OPTIONS = ["all_three", "skeletons_only", "somas_only", "combined"]
SAVE_EACH_RENDER_MODE_HTML = True


# Display-only orientation correction.
# This flips the shell, neurons, and somas together, only for plotting.
# It does not alter clusters, SWCs, UMAPs, or CSV exports.
RENDER_FLIP_X = False
RENDER_FLIP_Y = False
RENDER_FLIP_Z = True

# Real soma/cell-body rendering.
USE_NEUPRINT_SOMAS = True
NEUPRINT_SERVER = "https://neuprint-fish2.janelia.org"
NEUPRINT_DATASET = "fish2"
NEUPRINT_TOKEN_FALLBACK = None  # Prefer env var NEUPRINT_TOKEN or local neuprint_token.txt
SOMA_LOCATION_PRIORITY = ["somaLocation", "tosomaLocation", "rootLocation"]
# Soma/cell-body display mode:
#   "swc_local_blob" = preferred: render local SWC nodes/edges around soma/root with radius-scaled markers.
#   "sphere"         = metadata somaLocation/somaRadius as a simple sphere.
#   "marker"         = metadata/root as a simple point.
SOMA_RENDER_MODE = "swc_radius_mesh"
SOMA_MARKER_SIZE = 5
SOMA_SPHERE_OPACITY = 0.9
SOMA_SPHERE_RADIUS_SCALE = 1.0
SOMA_MARKER_SIZE_MIN = 4
SOMA_MARKER_SIZE_MAX = 12
SOMA_SPHERE_THETA_STEPS = 10
SOMA_SPHERE_PHI_STEPS = 8
FORCE_REFETCH_SOMAS = False

# Skeleton display settings.
# "lines" is fast and clean. "radius_tubes" is experimental and much heavier.
CELL_RENDER_MODE = "lines"
CELL_LINE_ALPHA = 0.95
CELL_TUBE_RADIUS_SCALE = 0.45
CELL_TUBE_MIN_RADIUS = 8.0
CELL_TUBE_MAX_RADIUS = 35.0
CELL_TUBE_MAX_EDGES_PER_NEURON = 1200
CELL_TUBE_SIDES = 6

# Soma/cell-body display settings.
# "swc_radius_mesh" is default: render a small surface mesh from the local SWC
# nodes/radii around the neuPrint soma/root anchor. This is the most realistic
# soma geometry available from neuPrint/SWC without external segmentation meshes.
# "local_body_mesh" tries body_meshes/<bodyId>.obj/.ply and clips near soma.
# "marker" and "sphere" are simple fallbacks.
SOMA_RADIUS_MESH_GRAPH_HOPS = 5
SOMA_RADIUS_MESH_EUCLIDEAN_RADIUS = 650.0
SOMA_RADIUS_MESH_MAX_NODES = 22
SOMA_RADIUS_MESH_MIN_NODES = 4
SOMA_RADIUS_MESH_RADIUS_SCALE = 1.15
SOMA_RADIUS_MESH_MIN_RADIUS = 25.0
SOMA_RADIUS_MESH_MAX_RADIUS = 230.0
SOMA_RADIUS_MESH_SPHERE_THETA = 9
SOMA_RADIUS_MESH_SPHERE_PHI = 7
SOMA_RADIUS_MESH_CYLINDER_SIDES = 9
SOMA_RADIUS_MESH_OPACITY = 0.92

# Optional true segmentation/body mesh hook.
# If you export per-body meshes later, place them in body_meshes/ as <bodyId>.obj or <bodyId>.ply.
# The notebook will clip vertices/faces near somaLocation/root and render that soma-region mesh.
LOCAL_BODY_MESH_DIR = PROJECT_DIR / "body_meshes"
PREFER_LOCAL_BODY_MESH_SOMA = True
LOCAL_BODY_MESH_SOMA_CLIP_RADIUS = 1200.0
LOCAL_BODY_MESH_MAX_FACES = 8000
LOCAL_BODY_MESH_OPACITY = 0.82

# Cluster colors: every cell and soma in a displayed cluster uses the same color.
CLUSTER_COLOR_PALETTE = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173",
    "#3182bd", "#31a354", "#756bb1", "#636363", "#e6550d",
]

# Output folders.
FIG_DIR = PLOT_OUT_ROOT / "figures"
RENDER_DIR = PLOT_OUT_ROOT / "cluster_renderings"
EXPORT_DIR = PLOT_OUT_ROOT / "cluster_id_exports"
SOMA_DIR = PLOT_OUT_ROOT / "neuprint_soma_metadata"
for p in [FIG_DIR, RENDER_DIR, EXPORT_DIR, SOMA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Previous outputs:", PREVIOUS_OUT_ROOT)
print("New plot-only outputs:", PLOT_OUT_ROOT)
print("Render flips: X=", RENDER_FLIP_X, "Y=", RENDER_FLIP_Y, "Z=", RENDER_FLIP_Z)


In [ ]:
# ============================================================
# 2. Load previous v14 clustering outputs
# ============================================================

def require_file(path, description="file"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {description}: {path}\n"
            "This plotting-only notebook expects that the v14 notebook has already completed."
        )
    return path


def load_dataset_from_previous_outputs(dataset_key, cfg):
    ds = dict(cfg)
    # Prefer the root assignment file, then fall back to cluster_id_exports.
    assign_path = PREVIOUS_OUT_ROOT / f"{dataset_key}_cluster_assignments.csv"
    if not assign_path.exists():
        assign_path = PREVIOUS_OUT_ROOT / "cluster_id_exports" / f"{dataset_key}_per_neuron_cluster_assignments.csv"
    require_file(assign_path, f"{dataset_key} assignments")
    assignments = pd.read_csv(assign_path)
    assignments["bodyId"] = assignments["bodyId"].astype(int)
    assignments["cluster"] = assignments["cluster"].astype(int)
    if "cluster_size" not in assignments.columns:
        sizes = assignments["cluster"].value_counts()
        assignments["cluster_size"] = assignments["cluster"].map(sizes).astype(int)
    if "cluster_name" not in assignments.columns:
        assignments["cluster_name"] = assignments["cluster"].map(lambda x: f"{dataset_key}_C{int(x):02d}")
    ds["assignments"] = assignments
    ds["body_ids"] = assignments["bodyId"].astype(int).tolist()
    ds["assignment_path"] = assign_path

    emb_path = PREVIOUS_OUT_ROOT / f"{dataset_key}_embeddings.csv"
    if emb_path.exists():
        emb = pd.read_csv(emb_path)
        emb["bodyId"] = emb["bodyId"].astype(int)
        ds["embeddings"] = emb
        ds["embedding_path"] = emb_path
    else:
        print(f"No embedding file found for {dataset_key}: {emb_path}")

    sweep_path = PREVIOUS_OUT_ROOT / f"{dataset_key}_cluster_sweep.csv"
    if sweep_path.exists():
        ds["sweep"] = pd.read_csv(sweep_path)

    return ds


datasets = {}
for key, cfg in DATASET_CONFIG.items():
    try:
        datasets[key] = load_dataset_from_previous_outputs(key, cfg)
        a = datasets[key]["assignments"]
        print(f"Loaded {key}: {len(a)} cells, {a['cluster'].nunique()} clusters from {datasets[key]['assignment_path']}")
    except FileNotFoundError as e:
        print(str(e))

if not datasets:
    raise RuntimeError("No previous dataset outputs could be loaded.")

# Quick display of cluster sizes.
for key, ds in datasets.items():
    print("\n", "="*80)
    print(ds["label"])
    display(ds["assignments"].groupby("cluster").size().sort_values(ascending=False).to_frame("n_cells").head(30))


In [ ]:
# ============================================================
# 3. Re-save UMAP/MDS figures from previous embeddings
# ============================================================

import matplotlib.colors as mcolors


def savefig_all(fig, path_base):
    path_base = Path(path_base)
    if SAVE_PNG:
        fig.savefig(path_base.with_suffix(".png"), dpi=FIG_DPI, bbox_inches="tight")
    if SAVE_PDF:
        fig.savefig(path_base.with_suffix(".pdf"), bbox_inches="tight")
    if SAVE_SVG:
        fig.savefig(path_base.with_suffix(".svg"), bbox_inches="tight")


def get_cluster_color(dataset_key, cluster):
    """Stable cluster color used across UMAPs and 3D cluster renders."""
    cluster = int(cluster)
    return CLUSTER_COLOR_PALETTE[(cluster - 1) % len(CLUSTER_COLOR_PALETTE)]


def get_cluster_color_map(dataset_key, clusters):
    clusters = sorted([int(c) for c in pd.Series(clusters).dropna().unique()])
    return {cl: get_cluster_color(dataset_key, cl) for cl in clusters}


def plot_embedding_from_loaded(ds, dataset_key, method="umap", label_points=False):
    if "embeddings" not in ds:
        print(f"Skipping {dataset_key} {method}: no embeddings loaded.")
        return None
    emb = ds["embeddings"].merge(ds["assignments"], on="bodyId", how="left")
    xcol, ycol = ("umap1", "umap2") if method == "umap" else ("mds1", "mds2")
    if xcol not in emb.columns or ycol not in emb.columns:
        print(f"Skipping {dataset_key} {method}: columns {xcol}/{ycol} not found.")
        return None

    clusters = sorted(emb["cluster"].dropna().astype(int).unique())
    color_map = get_cluster_color_map(dataset_key, clusters)

    fig, ax = plt.subplots(figsize=(6.8, 5.8), dpi=FIG_DPI)
    for cl in clusters:
        sub = emb.loc[emb["cluster"].astype(int) == cl]
        ax.scatter(
            sub[xcol], sub[ycol],
            color=color_map[cl],
            s=46,
            alpha=0.92,
            edgecolor="white",
            linewidth=0.35,
            label=f"C{cl:02d} (n={len(sub)})",
        )
    if label_points and len(emb) <= 150:
        for _, r in emb.iterrows():
            ax.text(r[xcol], r[ycol], str(int(r["bodyId"])), fontsize=4, alpha=0.65)
    ax.set_title(f"{ds['label']} | {method.upper()} | {emb['cluster'].nunique()} clusters")
    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol)
    ax.set_aspect("equal", adjustable="datalim")
    if len(clusters) <= 18:
        ax.legend(frameon=False, fontsize=6, markerscale=0.7, loc="best")
    fig.tight_layout()
    savefig_all(fig, FIG_DIR / f"{dataset_key}_{method}_clusters_plotonly_v16_clusterColors")
    display(fig)
    plt.close(fig)
    return emb

for key, ds in datasets.items():
    plot_embedding_from_loaded(ds, key, method="umap", label_points=False)
    plot_embedding_from_loaded(ds, key, method="mds", label_points=False)

# Summary figure from loaded assignments.
summary_rows = []
for key, ds in datasets.items():
    a = ds["assignments"]
    sizes = a["cluster"].value_counts()
    summary_rows.append({
        "dataset_key": key,
        "dataset": ds["label"],
        "n_cells_clustered": len(a),
        "n_clusters_total": int(sizes.size),
        "n_multi_neuron_clusters": int((sizes >= 2).sum()),
        "n_singletons": int((sizes == 1).sum()),
        "largest_cluster": int(sizes.max()),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(PLOT_OUT_ROOT / "summary_cluster_counts_loaded_from_v14.csv", index=False)
display(summary_df)

if not summary_df.empty:
    fig, ax = plt.subplots(figsize=(7.4, 4.8), dpi=FIG_DPI)
    x = np.arange(len(summary_df))
    w = 0.25
    ax.bar(x - w, summary_df["n_clusters_total"], width=w, label="total clusters")
    ax.bar(x, summary_df["n_multi_neuron_clusters"], width=w, label="multi-neuron clusters")
    ax.bar(x + w, summary_df["n_singletons"], width=w, label="singletons")
    ax.set_xticks(x)
    ax.set_xticklabels(summary_df["dataset"], rotation=20, ha="right")
    ax.set_ylabel("Count")
    ax.set_title("Morphology clustering summary — loaded from v14")
    ax.legend(frameon=False)
    fig.tight_layout()
    savefig_all(fig, FIG_DIR / "summary_cluster_counts_loaded_from_v14")
    display(fig)
    plt.close(fig)


In [ ]:
# ============================================================
# 4. Excel-friendly cluster ID exports from loaded assignments
# ============================================================

try:
    import openpyxl  # noqa
    HAS_OPENPYXL = True
except Exception:
    HAS_OPENPYXL = False


def export_loaded_cluster_ids(ds, dataset_key):
    a = ds["assignments"].copy().sort_values(["cluster", "bodyId"])
    a["dataset"] = dataset_key
    a["dataset_label"] = ds["label"]
    a["cluster_name"] = a["cluster"].map(lambda x: f"{dataset_key}_C{int(x):02d}")
    sizes = a["cluster"].value_counts()
    a["cluster_size"] = a["cluster"].map(sizes).astype(int)

    per_neuron_path = EXPORT_DIR / f"{dataset_key}_per_neuron_cluster_assignments_loaded.csv"
    a.to_csv(per_neuron_path, index=False)

    rows = []
    for cl, sub in a.groupby("cluster"):
        ids = sub["bodyId"].astype(int).tolist()
        rows.append({
            "dataset": dataset_key,
            "dataset_label": ds["label"],
            "cluster": int(cl),
            "cluster_name": f"{dataset_key}_C{int(cl):02d}",
            "n_cells": len(ids),
            "bodyIds_comma": ",".join(map(str, ids)),
            "bodyIds_semicolon": ";".join(map(str, ids)),
        })
    cluster_df = pd.DataFrame(rows).sort_values(["n_cells", "cluster"], ascending=[False, True])
    cluster_path = EXPORT_DIR / f"{dataset_key}_cluster_id_lists_loaded.csv"
    cluster_df.to_csv(cluster_path, index=False)

    print(f"Saved: {per_neuron_path}")
    print(f"Saved: {cluster_path}")

    if HAS_OPENPYXL:
        xlsx = EXPORT_DIR / f"{dataset_key}_cluster_ids_loaded.xlsx"
        with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
            a.to_excel(writer, sheet_name="per_neuron", index=False)
            cluster_df.to_excel(writer, sheet_name="per_cluster", index=False)
        print(f"Saved: {xlsx}")

    return cluster_df

all_cluster_summaries = []
for key, ds in datasets.items():
    all_cluster_summaries.append(export_loaded_cluster_ids(ds, key))

combined_cluster_summary = pd.concat(all_cluster_summaries, ignore_index=True)
combined_path = EXPORT_DIR / "combined_cluster_id_lists_loaded.csv"
combined_cluster_summary.to_csv(combined_path, index=False)
print(f"Saved: {combined_path}")
display(combined_cluster_summary)


## 4b. Recluster selected raphe without rerunning NBLAST

This section uses the cached selected-raphe NBLAST distance matrix from the previous full analysis. It tries several cluster numbers, saves a UMAP for each, and then sets one chosen `k` as the active clustering for all downstream checks and figure-ready 3D exports.

In [ ]:
# ============================================================
# 4b. Recluster selected raphe from existing NBLAST distance matrix
# ============================================================
# This cell does NOT rerun NBLAST.
# It reloads the cached selected-raphe NBLAST distance matrix, recuts the
# hierarchy into different cluster numbers, saves UMAPs/assignments for each k,
# and lets you choose which k becomes the active clustering for all downstream
# cluster-checking and figure-ready 3D cells.

from scipy.cluster.hierarchy import linkage, cut_tree
from scipy.spatial.distance import squareform

# ----------------------------
# User settings
# ----------------------------

# Try several cluster numbers. Add/remove values here.
SELECTED_RAPHE_RECLUSTER_K_VALUES = [7, 8, 9, 10, 11, 12, 14, 16]

# Pick the one you want downstream cells to use.
# After looking at the UMAPs, change this and rerun this cell.
SELECTED_RAPHE_ACTIVE_K = 10

# For NBLAST-like distances, average linkage is usually a reasonable default.
SELECTED_RAPHE_LINKAGE_METHOD = "average"

# Highlight checked IDs on every candidate UMAP.
try:
    SELECTED_RAPHE_IDS_TO_HIGHLIGHT = [int(x) for x in CELLS_TO_CHECK]
except Exception:
    SELECTED_RAPHE_IDS_TO_HIGHLIGHT = [
        100791363,
        102967931,
        106511646,
        105165440,
        100427087,
        100312010,
        103164598,
        103159060,
        101079563,
        103160069,
        100098958,
        100668438,
        100528051,
        100463138,
        109988161,
    ]

RECLUSTER_POINT_SIZE = 52
RECLUSTER_HIGHLIGHT_SIZE = 170
RECLUSTER_FIG_DPI = 350

SELECTED_RAPHE_RECLUSTER_DIR = PLOT_OUT_ROOT / "selected_raphe_recluster_existing_nblast"
SELECTED_RAPHE_RECLUSTER_DIR.mkdir(parents=True, exist_ok=True)


# ----------------------------
# Distance-matrix loading
# ----------------------------

def _coerce_square_dataframe(obj, source_path=None):
    """Convert a loaded object into a square bodyId x bodyId distance DataFrame."""
    if isinstance(obj, pd.DataFrame):
        df = obj.copy()
    elif isinstance(obj, np.ndarray):
        arr = np.asarray(obj)
        if arr.ndim != 2 or arr.shape[0] != arr.shape[1]:
            raise ValueError("Numpy object is not a square matrix.")
        df = pd.DataFrame(arr)
    else:
        raise TypeError(f"Cannot interpret object of type {type(obj)} as a distance matrix.")

    # If a CSV was saved with an unnamed first index column, pandas can load it correctly
    # via index_col=0. But if not, this tries to recover bodyId columns.
    if "bodyId" in df.columns and df.shape[1] == df.shape[0] + 1:
        df = df.set_index("bodyId")

    # Remove accidental unnamed index columns.
    unnamed = [c for c in df.columns if str(c).startswith("Unnamed")]
    if unnamed and df.shape[1] == df.shape[0] + len(unnamed):
        df = df.drop(columns=unnamed)

    # Coerce labels to ints when possible. If labels are 0..n-1, replace with current IDs.
    try:
        df.index = df.index.astype(int)
        df.columns = df.columns.astype(int)
    except Exception:
        current_ids = datasets["selected_raphe"]["assignments"]["bodyId"].astype(int).tolist()
        if df.shape[0] == len(current_ids) and df.shape[1] == len(current_ids):
            df.index = current_ids
            df.columns = current_ids
        else:
            raise ValueError(
                f"Could not convert distance matrix labels to body IDs for {source_path}."
            )

    if df.shape[0] != df.shape[1]:
        raise ValueError(f"Distance matrix is not square: {df.shape} from {source_path}")

    # Keep only currently loaded selected-raphe cells.
    current_ids = datasets["selected_raphe"]["assignments"]["bodyId"].astype(int).tolist()
    keep_ids = [bid for bid in current_ids if bid in df.index and bid in df.columns]

    if len(keep_ids) < 2:
        raise ValueError(
            f"Distance matrix from {source_path} has too few selected-raphe IDs overlapping current assignments."
        )

    missing = sorted(set(current_ids) - set(keep_ids))
    if missing:
        print(f"Warning: {len(missing)} selected-raphe assignment IDs missing from distance matrix.")
        print("First missing IDs:", missing[:20])

    df = df.loc[keep_ids, keep_ids].astype(float)
    df = (df + df.T) / 2.0
    np.fill_diagonal(df.values, 0.0)

    if not np.isfinite(df.values).all():
        bad = np.sum(~np.isfinite(df.values))
        raise ValueError(f"Distance matrix contains {bad} non-finite values: {source_path}")

    return df


def _load_matrix_path(path):
    """Load one candidate matrix path."""
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix in [".pkl", ".pickle"]:
        obj = pd.read_pickle(path)
        return _coerce_square_dataframe(obj, path)

    if suffix == ".csv":
        # First try index_col=0, then no index if needed.
        try:
            df = pd.read_csv(path, index_col=0)
            return _coerce_square_dataframe(df, path)
        except Exception as e1:
            df = pd.read_csv(path)
            try:
                return _coerce_square_dataframe(df, path)
            except Exception:
                raise e1

    if suffix == ".npy":
        obj = np.load(path, allow_pickle=True)
        return _coerce_square_dataframe(obj, path)

    if suffix == ".npz":
        z = np.load(path, allow_pickle=True)
        arrays = {k: z[k] for k in z.files}
        matrix_key = None
        for k, arr in arrays.items():
            if isinstance(arr, np.ndarray) and arr.ndim == 2 and arr.shape[0] == arr.shape[1]:
                matrix_key = k
                break
        if matrix_key is None:
            raise ValueError(f"No square matrix found in {path}")
        arr = arrays[matrix_key]
        df = _coerce_square_dataframe(arr, path)
        for id_key in ["bodyIds", "body_ids", "ids", "bodyId", "index"]:
            if id_key in arrays and len(arrays[id_key]) == df.shape[0]:
                ids = [int(x) for x in arrays[id_key]]
                df.index = ids
                df.columns = ids
                break
        return _coerce_square_dataframe(df, path)

    raise ValueError(f"Unsupported matrix file type: {path}")


def _candidate_selected_raphe_distance_paths():
    """Return likely cached selected-raphe distance matrix paths, best first."""
    candidates = []

    explicit = [
        PREVIOUS_OUT_ROOT / "nblast_cache" / "selected_raphe" / "selected_raphe_fishfuncem_distance_rank.pkl",
        PREVIOUS_OUT_ROOT / "nblast_cache" / "selected_raphe" / "selected_raphe_fishfuncem_distance_raw_inverse.pkl",
        PREVIOUS_OUT_ROOT / "nblast_cache" / "selected_raphe" / "selected_raphe_fishfuncem_distance_rank.csv",
        PREVIOUS_OUT_ROOT / "nblast_cache" / "selected_raphe" / "selected_raphe_fishfuncem_distance_raw_inverse.csv",
        PREVIOUS_OUT_ROOT / "selected_raphe_distance_matrix.pkl",
        PREVIOUS_OUT_ROOT / "selected_raphe_distance_matrix.csv",
        PREVIOUS_OUT_ROOT / "selected_raphe_nblast_distance.pkl",
        PREVIOUS_OUT_ROOT / "selected_raphe_nblast_distance.csv",
    ]
    candidates.extend([p for p in explicit if p.exists()])

    # Recursive fallback search. Prefer files with both selected_raphe and distance/dist.
    patterns = [
        "**/*selected*raphe*distance*.pkl",
        "**/*selected*raphe*distance*.csv",
        "**/*selected*raphe*dist*.pkl",
        "**/*selected*raphe*dist*.csv",
        "**/*selected*nblast*distance*.pkl",
        "**/*selected*nblast*distance*.csv",
        "**/*selected*nblast*dist*.pkl",
        "**/*selected*nblast*dist*.csv",
    ]
    for pat in patterns:
        candidates.extend(PREVIOUS_OUT_ROOT.glob(pat))

    # Deduplicate and avoid obvious non-matrix files.
    seen = set()
    out = []
    bad_words = ["cluster", "assignment", "sweep", "summary", "embedding", "umap", "mds", "ids", "id_lists"]
    for p in candidates:
        p = Path(p)
        key = str(p.resolve())
        if key in seen:
            continue
        seen.add(key)
        name = p.name.lower()
        if any(w in name for w in bad_words):
            continue
        out.append(p)

    return out


def load_selected_raphe_nblast_distance_matrix():
    """Load the cached selected-raphe NBLAST distance matrix without recomputing NBLAST."""
    paths = _candidate_selected_raphe_distance_paths()

    print("Candidate selected-raphe distance matrices:")
    for p in paths[:20]:
        print("  ", p)
    if len(paths) > 20:
        print(f"  ... and {len(paths) - 20} more")

    errors = []
    for p in paths:
        try:
            dist = _load_matrix_path(p)
            print(f"\nLoaded selected-raphe distance matrix from: {p}")
            print(f"Shape: {dist.shape[0]} x {dist.shape[1]}")
            return dist, p
        except Exception as e:
            errors.append((p, repr(e)))

    print("\nFailed candidate loads:")
    for p, err in errors[:15]:
        print("  ", p)
        print("     ", err)

    raise FileNotFoundError(
        "Could not locate/load a cached selected-raphe NBLAST distance matrix. "
        "Run the original NBLAST notebook first, or point the candidate list to the saved distance file."
    )


# ----------------------------
# Reclustering + UMAP plotting
# ----------------------------

def make_linkage_from_selected_raphe_distance(dist, method=SELECTED_RAPHE_LINKAGE_METHOD):
    arr = dist.to_numpy(float)
    arr = (arr + arr.T) / 2.0
    np.fill_diagonal(arr, 0.0)
    condensed = squareform(arr, checks=False)
    Z = linkage(condensed, method=method)
    return Z


def assign_selected_raphe_clusters_for_k(dist, Z, k):
    ids = [int(x) for x in dist.index.tolist()]
    labels = cut_tree(Z, n_clusters=int(k)).ravel().astype(int) + 1

    out = pd.DataFrame({"bodyId": ids, "cluster": labels})

    # Renumber by cluster size descending, so C01 is largest.
    old_sizes = out["cluster"].value_counts()
    old_order = old_sizes.sort_values(ascending=False).index.tolist()
    old_to_new = {old: i + 1 for i, old in enumerate(old_order)}
    out["cluster"] = out["cluster"].map(old_to_new).astype(int)

    sizes = out["cluster"].value_counts()
    out["cluster_size"] = out["cluster"].map(sizes).astype(int)
    out["cluster_name"] = out["cluster"].map(lambda c: f"selected_raphe_k{int(k):02d}_C{int(c):02d}")
    out["recluster_k"] = int(k)
    out["clustering_source"] = "existing_selected_raphe_nblast_distance"

    return out.sort_values(["cluster", "bodyId"]).reset_index(drop=True)


def _selected_raphe_embedding_for_recluster(method="umap"):
    ds = datasets["selected_raphe"]
    if "embeddings" not in ds:
        raise RuntimeError("datasets['selected_raphe'] has no embeddings loaded.")

    emb_obj = ds["embeddings"]
    if isinstance(emb_obj, dict):
        emb = emb_obj.get(method)
        if emb is None:
            raise RuntimeError(f"No {method} embedding found in ds['embeddings'] dict.")
        emb = emb.copy()
    else:
        emb = emb_obj.copy()

    emb["bodyId"] = emb["bodyId"].astype(int)
    possible_x = [f"{method}1", f"{method}_1", f"{method.upper()}1", f"{method.upper()}_1"]
    possible_y = [f"{method}2", f"{method}_2", f"{method.upper()}2", f"{method.upper()}_2"]
    xcol = next((c for c in possible_x if c in emb.columns), None)
    ycol = next((c for c in possible_y if c in emb.columns), None)
    if xcol is None or ycol is None:
        raise RuntimeError(f"Could not find {method} coordinate columns in selected_raphe embedding.")

    return emb, xcol, ycol


def plot_selected_raphe_recluster_embedding(k, assignments, method="umap", save=True, show=True):
    emb, xcol, ycol = _selected_raphe_embedding_for_recluster(method)
    plot_df = emb.merge(assignments, on="bodyId", how="inner")

    clusters = sorted(plot_df["cluster"].dropna().astype(int).unique())
    color_map = get_cluster_color_map("selected_raphe", clusters)

    fig, ax = plt.subplots(figsize=(8.4, 6.9), dpi=RECLUSTER_FIG_DPI)

    for cl in clusters:
        sub = plot_df.loc[plot_df["cluster"].astype(int) == cl]
        ax.scatter(
            sub[xcol],
            sub[ycol],
            s=RECLUSTER_POINT_SIZE,
            color=color_map[cl],
            alpha=0.92,
            edgecolor="white",
            linewidth=0.45,
            label=f"C{cl:02d} (n={len(sub)})",
        )

    # Yellow stars for IDs of interest.
    highlight_ids = sorted(set(int(x) for x in SELECTED_RAPHE_IDS_TO_HIGHLIGHT))
    if highlight_ids:
        h = plot_df.loc[plot_df["bodyId"].astype(int).isin(highlight_ids)].copy()
        if len(h):
            ax.scatter(
                h[xcol],
                h[ycol],
                s=RECLUSTER_HIGHLIGHT_SIZE,
                marker="*",
                color="yellow",
                edgecolor="black",
                linewidth=0.8,
                zorder=10,
                label="checked IDs",
            )
            for _, r in h.iterrows():
                ax.text(
                    r[xcol],
                    r[ycol],
                    str(int(r["bodyId"])),
                    fontsize=5.5,
                    ha="left",
                    va="bottom",
                    color="black",
                    zorder=11,
                )
        missing_highlights = sorted(set(highlight_ids) - set(h["bodyId"].astype(int).tolist()))
        if missing_highlights:
            print(f"k={k} {method}: highlighted IDs not found in selected_raphe embedding:", missing_highlights)

    ax.set_title(f"Selected raphe neurons | {method.upper()} | reclustered k={int(k)}")
    ax.set_xlabel(xcol)
    ax.set_ylabel(ycol)
    ax.set_aspect("equal", adjustable="datalim")
    if len(clusters) <= 18:
        ax.legend(frameon=False, fontsize=7, markerscale=0.85, loc="best")

    fig.tight_layout()

    if save:
        out_base = SELECTED_RAPHE_RECLUSTER_DIR / f"selected_raphe_recluster_k{int(k):02d}_{method}"
        if SAVE_PNG:
            fig.savefig(out_base.with_suffix(".png"), dpi=RECLUSTER_FIG_DPI, bbox_inches="tight")
        if SAVE_PDF:
            fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
        if SAVE_SVG:
            fig.savefig(out_base.with_suffix(".svg"), bbox_inches="tight")
        print(f"Saved {method.upper()} for k={k}: {out_base}")

    if show:
        display(fig)

    plt.close(fig)
    return plot_df


def run_selected_raphe_recluster_sweep():
    dist, dist_path = load_selected_raphe_nblast_distance_matrix()
    Z = make_linkage_from_selected_raphe_distance(dist)

    variants = {}
    summary_rows = []

    for k in SELECTED_RAPHE_RECLUSTER_K_VALUES:
        k = int(k)
        assignments_k = assign_selected_raphe_clusters_for_k(dist, Z, k)
        variants[k] = assignments_k

        assign_path = SELECTED_RAPHE_RECLUSTER_DIR / f"selected_raphe_recluster_k{k:02d}_assignments.csv"
        assignments_k.to_csv(assign_path, index=False)

        cluster_rows = []
        for cl, sub in assignments_k.groupby("cluster"):
            body_ids = sub["bodyId"].astype(int).sort_values().tolist()
            cluster_rows.append({
                "k": k,
                "cluster": int(cl),
                "cluster_name": f"selected_raphe_k{k:02d}_C{int(cl):02d}",
                "n_cells": len(body_ids),
                "bodyIds_semicolon": ";".join(map(str, body_ids)),
                "bodyIds_comma": ",".join(map(str, body_ids)),
            })
        cluster_list = pd.DataFrame(cluster_rows)
        cluster_list_path = SELECTED_RAPHE_RECLUSTER_DIR / f"selected_raphe_recluster_k{k:02d}_cluster_id_lists.csv"
        cluster_list.to_csv(cluster_list_path, index=False)

        sizes = assignments_k["cluster"].value_counts()
        summary_rows.append({
            "k": k,
            "n_cells": len(assignments_k),
            "n_clusters": int(assignments_k["cluster"].nunique()),
            "largest_cluster": int(sizes.max()),
            "smallest_cluster": int(sizes.min()),
            "n_singletons": int((sizes == 1).sum()),
            "n_clusters_ge_2": int((sizes >= 2).sum()),
            "n_clusters_ge_4": int((sizes >= 4).sum()),
            "distance_matrix_path": str(dist_path),
            "assignments_csv": str(assign_path),
            "cluster_id_lists_csv": str(cluster_list_path),
        })

        plot_selected_raphe_recluster_embedding(k, assignments_k, method="umap", save=True, show=True)

    summary = pd.DataFrame(summary_rows)
    summary_path = SELECTED_RAPHE_RECLUSTER_DIR / "selected_raphe_recluster_k_sweep_summary.csv"
    summary.to_csv(summary_path, index=False)

    print("\nSaved recluster summary:", summary_path)
    display(summary)

    return dist, Z, variants, summary


def apply_selected_raphe_recluster_k(k=SELECTED_RAPHE_ACTIVE_K):
    """
    Make one k the active selected_raphe clustering for all downstream cells.
    This overwrites datasets['selected_raphe']['assignments'], but keeps a backup.
    """
    k = int(k)

    if "selected_raphe_recluster_variants" not in globals():
        raise RuntimeError("Run run_selected_raphe_recluster_sweep() first.")

    if k not in selected_raphe_recluster_variants:
        raise KeyError(
            f"k={k} was not generated. Available k values: {sorted(selected_raphe_recluster_variants.keys())}"
        )

    new_assignments = selected_raphe_recluster_variants[k].copy()
    new_assignments["bodyId"] = new_assignments["bodyId"].astype(int)
    new_assignments["cluster"] = new_assignments["cluster"].astype(int)

    ds = datasets["selected_raphe"]
    if "assignments_original_v14" not in ds:
        ds["assignments_original_v14"] = ds["assignments"].copy()

    ds["assignments"] = new_assignments
    ds["active_recluster_k"] = k
    ds["label"] = f"Selected raphe neurons | reclustered k={k}"

    active_path = SELECTED_RAPHE_RECLUSTER_DIR / "ACTIVE_selected_raphe_assignments.csv"
    new_assignments.to_csv(active_path, index=False)

    print("=" * 90)
    print(f"ACTIVE selected_raphe clustering is now k={k}")
    print(f"Saved active assignments: {active_path}")
    print("Downstream cluster checks, UMAP overlays, and fixed 3D figure-ready cells now use this clustering.")
    print("=" * 90)

    cluster_summary = (
        new_assignments.groupby("cluster")
        .size()
        .rename("n_cells")
        .reset_index()
        .sort_values("cluster")
    )
    display(cluster_summary)

    return new_assignments


# ----------------------------
# Run sweep + activate chosen k
# ----------------------------

selected_raphe_recluster_dist, selected_raphe_recluster_Z, selected_raphe_recluster_variants, selected_raphe_recluster_summary = (
    run_selected_raphe_recluster_sweep()
)

selected_raphe_active_assignments = apply_selected_raphe_recluster_k(SELECTED_RAPHE_ACTIVE_K)


In [ ]:
# ============================================================
# 5. SWC parser + display-only flipped coordinate transform
# ============================================================

SWC_COLS = ["node_id", "type", "x", "y", "z", "radius", "parent"]


def body_id_from_path(path):
    m = re.search(r"(\d+)", Path(path).stem)
    return int(m.group(1)) if m else None


def read_swc(path):
    rows = []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 7:
                continue
            try:
                rows.append([
                    int(float(parts[0])), int(float(parts[1])),
                    float(parts[2]), float(parts[3]), float(parts[4]),
                    float(parts[5]), int(float(parts[6])),
                ])
            except Exception:
                pass
    return pd.DataFrame(rows, columns=SWC_COLS)


def swc_edges(df):
    if df is None or df.empty:
        return []
    ids = set(df["node_id"].astype(int))
    edges = []
    for _, r in df.iterrows():
        p = int(r["parent"])
        n = int(r["node_id"])
        if p in ids and p != -1:
            edges.append((p, n))
    return edges


def estimate_soma_from_swc(df):
    if df is None or df.empty:
        return None
    # Prefer SWC type 1 soma node if available.
    soma_nodes = df.loc[df["type"].astype(int) == 1]
    if len(soma_nodes):
        r = soma_nodes.iloc[0]
        return np.array([r["x"], r["y"], r["z"]], dtype=float)
    # Then root.
    root = df.loc[(df["parent"].astype(int) == -1)]
    if len(root):
        r = root.iloc[0]
        return np.array([r["x"], r["y"], r["z"]], dtype=float)
    # Then largest-radius node.
    if "radius" in df.columns and df["radius"].notna().any():
        r = df.loc[df["radius"].astype(float).idxmax()]
        return np.array([r["x"], r["y"], r["z"]], dtype=float)
    return df[["x", "y", "z"]].median().values.astype(float)


def swc_path_for_body(body_id, swc_dir):
    swc_dir = Path(swc_dir)
    direct = swc_dir / f"{int(body_id)}.swc"
    if direct.exists():
        return direct
    matches = list(swc_dir.glob(f"*{int(body_id)}*.swc"))
    return matches[0] if matches else None


def swc_segments_for_body(body_id, swc_dir):
    path = swc_path_for_body(body_id, swc_dir)
    if path is None:
        return [], None
    df = read_swc(path)
    if df.empty:
        return [], None
    coords = df.set_index("node_id")[["x", "y", "z"]]
    segs = []
    for p, n in swc_edges(df):
        try:
            a = coords.loc[p].values.astype(float)
            b = coords.loc[n].values.astype(float)
            segs.append((a, b))
        except Exception:
            pass
    soma = estimate_soma_from_swc(df)
    return segs, soma

# Load the shell from the previous v14 run.
SHELL_NPZ = PREVIOUS_OUT_ROOT / "outer_brain_shell" / "outer_brain_shell_voxel_union.npz"
if SHELL_NPZ.exists():
    shell_data = np.load(SHELL_NPZ)
    outer_shell_vertices = shell_data["vertices"]
    outer_shell_faces = shell_data["faces"].astype(int)
    print("Loaded outer shell:", SHELL_NPZ, outer_shell_vertices.shape, outer_shell_faces.shape)
else:
    outer_shell_vertices = None
    outer_shell_faces = None
    print("No previous outer shell found:", SHELL_NPZ)


# ------------------------------------------------------------
# Clean brain shell mesh: remove disconnected ROI/mask blobs
# ------------------------------------------------------------
# The raw shell built from all neuPrint ROI meshes can include small disconnected
# components outside the brain. For figure-ready rendering we keep only the
# largest connected mesh component, which gives a cleaner translucent brain
# outline like the reference figure.
SHELL_KEEP_LARGEST_COMPONENT_ONLY = True
SHELL_MIN_COMPONENT_FACE_FRACTION = 0.02


def clean_outer_shell_mesh_largest_component(vertices, faces, min_face_fraction=SHELL_MIN_COMPONENT_FACE_FRACTION):
    """
    Keep only the largest connected component of a triangular mesh.

    Connectivity is defined by faces sharing vertices. This removes small
    disconnected ROI blobs / masks outside the main brain shell.
    """
    if vertices is None or faces is None:
        return vertices, faces, {"status": "missing"}

    vertices = np.asarray(vertices, dtype=float)
    faces = np.asarray(faces, dtype=int)

    if len(vertices) == 0 or len(faces) == 0:
        return vertices, faces, {"status": "empty"}

    from collections import defaultdict, deque

    vertex_to_faces = defaultdict(list)
    for fi, tri in enumerate(faces):
        for v in tri:
            vertex_to_faces[int(v)].append(fi)

    seen = np.zeros(len(faces), dtype=bool)
    components = []

    for start in range(len(faces)):
        if seen[start]:
            continue
        q = deque([start])
        seen[start] = True
        comp = []
        while q:
            fi = q.popleft()
            comp.append(fi)
            for v in faces[fi]:
                for nb in vertex_to_faces[int(v)]:
                    if not seen[nb]:
                        seen[nb] = True
                        q.append(nb)
        components.append(np.asarray(comp, dtype=int))

    components = sorted(components, key=len, reverse=True)
    largest = components[0]
    min_faces = max(1, int(len(faces) * float(min_face_fraction)))

    # Default: largest component only. This is what removes the little blobs.
    keep_faces = largest

    kept_faces = faces[keep_faces]
    used_vertices = np.unique(kept_faces.ravel())
    old_to_new = {int(old): i for i, old in enumerate(used_vertices)}
    remapped_faces = np.vectorize(old_to_new.get)(kept_faces).astype(int)
    cleaned_vertices = vertices[used_vertices]

    info = {
        "status": "cleaned",
        "n_components": len(components),
        "original_vertices": int(len(vertices)),
        "original_faces": int(len(faces)),
        "kept_vertices": int(len(cleaned_vertices)),
        "kept_faces": int(len(remapped_faces)),
        "largest_component_faces": int(len(largest)),
        "min_faces_threshold": int(min_faces),
        "removed_faces": int(len(faces) - len(remapped_faces)),
        "component_face_counts_top10": [int(len(c)) for c in components[:10]],
    }

    print("Brain shell cleaning:")
    print(f"  components found: {info['n_components']}")
    print(f"  original faces: {info['original_faces']:,}")
    print(f"  kept faces:     {info['kept_faces']:,}")
    print(f"  removed faces:  {info['removed_faces']:,}")
    print(f"  top component face counts: {info['component_face_counts_top10']}")

    return cleaned_vertices, remapped_faces, info


if SHELL_KEEP_LARGEST_COMPONENT_ONLY and outer_shell_vertices is not None and outer_shell_faces is not None:
    outer_shell_vertices, outer_shell_faces, SHELL_CLEANING_INFO = clean_outer_shell_mesh_largest_component(
        outer_shell_vertices,
        outer_shell_faces,
    )
else:
    SHELL_CLEANING_INFO = {"status": "not_cleaned"}


def compute_render_center():
    if outer_shell_vertices is not None and len(outer_shell_vertices):
        return np.nanmean(outer_shell_vertices, axis=0)
    pts = []
    for ds in datasets.values():
        swc_dir = ds.get("raw_swc_dir")
        for bid in ds.get("body_ids", [])[:50]:
            path = swc_path_for_body(bid, swc_dir)
            if path:
                df = read_swc(path)
                if len(df):
                    pts.append(df[["x", "y", "z"]].values)
    if pts:
        return np.nanmean(np.vstack(pts), axis=0)
    return np.array([0.0, 0.0, 0.0])

RENDER_CENTER = compute_render_center()
print("Render center:", RENDER_CENTER)


def transform_points_for_render(points):
    """Apply display-only reflection around RENDER_CENTER."""
    if points is None:
        return None
    arr = np.asarray(points, dtype=float).copy()
    was_1d = arr.ndim == 1
    if was_1d:
        arr = arr[None, :]
    if arr.shape[1] < 3:
        return points
    center = np.asarray(RENDER_CENTER, dtype=float)
    if RENDER_FLIP_X:
        arr[:, 0] = 2 * center[0] - arr[:, 0]
    if RENDER_FLIP_Y:
        arr[:, 1] = 2 * center[1] - arr[:, 1]
    if RENDER_FLIP_Z:
        arr[:, 2] = 2 * center[2] - arr[:, 2]
    return arr[0] if was_1d else arr

outer_shell_vertices_render = transform_points_for_render(outer_shell_vertices) if outer_shell_vertices is not None else None
print("Render flips applied: X=", RENDER_FLIP_X, "Y=", RENDER_FLIP_Y, "Z=", RENDER_FLIP_Z)


In [ ]:
# ============================================================
# 6. neuPrint soma metadata: real somaLocation/somaRadius when available
# ============================================================

def get_neuprint_token():
    token = os.environ.get("NEUPRINT_TOKEN", "")
    if token:
        return token
    token_file = PROJECT_DIR / "neuprint_token.txt"
    if token_file.exists():
        token = token_file.read_text().strip()
        if token:
            os.environ["NEUPRINT_TOKEN"] = token
            return token
    if NEUPRINT_TOKEN_FALLBACK:
        os.environ["NEUPRINT_TOKEN"] = NEUPRINT_TOKEN_FALLBACK
        return NEUPRINT_TOKEN_FALLBACK
    return ""


def make_neuprint_client_or_none():
    if not HAS_NEUPRINT:
        return None
    token = get_neuprint_token()
    if not token:
        print("No neuPrint token found. Soma metadata will fall back to SWC estimates.")
        return None
    try:
        return Client(NEUPRINT_SERVER, dataset=NEUPRINT_DATASET, token=token)
    except Exception as e:
        print("Could not create neuPrint client. Soma metadata will fall back to SWC estimates.")
        print(repr(e))
        return None

c = make_neuprint_client_or_none()


def parse_neuprint_location_value(v):
    if v is None:
        return None
    try:
        if isinstance(v, float) and np.isnan(v):
            return None
    except Exception:
        pass
    if isinstance(v, str):
        nums = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", v)
        if len(nums) >= 3:
            return np.array([float(nums[0]), float(nums[1]), float(nums[2])], dtype=float)
        return None
    if isinstance(v, (list, tuple, np.ndarray)) and len(v) >= 3:
        return np.array([float(v[0]), float(v[1]), float(v[2])], dtype=float)
    if isinstance(v, dict):
        if all(k in v for k in ["x", "y", "z"]):
            return np.array([float(v["x"]), float(v["y"]), float(v["z"])], dtype=float)
        if "coordinates" in v and len(v["coordinates"]) >= 3:
            vv = v["coordinates"]
            return np.array([float(vv[0]), float(vv[1]), float(vv[2])], dtype=float)
    if all(hasattr(v, k) for k in ["x", "y", "z"]):
        return np.array([float(v.x), float(v.y), float(v.z)], dtype=float)
    return None


def fetch_neuprint_soma_metadata(body_ids, label="all_datasets"):
    body_ids = sorted(set(int(x) for x in body_ids if pd.notna(x)))
    if not body_ids or c is None or not USE_NEUPRINT_SOMAS:
        return pd.DataFrame()
    out_csv = SOMA_DIR / f"{label}_soma_metadata.csv"
    if out_csv.exists() and not FORCE_REFETCH_SOMAS:
        df = pd.read_csv(out_csv)
        print(f"Loaded cached soma metadata: {out_csv} | rows={len(df)}")
        return df
    rows = []
    batch_size = 300
    for i in tqdm(range(0, len(body_ids), batch_size), desc=f"Fetch neuPrint soma metadata {label}"):
        batch = body_ids[i:i+batch_size]
        try:
            neurons, _ = fetch_neurons(NC(bodyId=batch), client=c)
        except Exception as e:
            print(f"Soma metadata fetch failed for batch {i//batch_size}: {repr(e)}")
            continue
        if neurons is None or neurons.empty:
            continue
        for _, r in neurons.iterrows():
            bid = int(r["bodyId"])
            loc = None
            source = None
            for col in SOMA_LOCATION_PRIORITY:
                if col in neurons.columns:
                    loc = parse_neuprint_location_value(r.get(col, None))
                    if loc is not None:
                        source = col
                        break
            radius = np.nan
            for rad_col in ["somaRadius", "radius"]:
                if rad_col in neurons.columns:
                    try:
                        radius = float(r.get(rad_col))
                    except Exception:
                        radius = np.nan
                    break
            rows.append({
                "bodyId": bid,
                "soma_x": np.nan if loc is None else loc[0],
                "soma_y": np.nan if loc is None else loc[1],
                "soma_z": np.nan if loc is None else loc[2],
                "soma_radius": radius,
                "soma_source": "neuprint_" + str(source) if source else "neuprint_missing",
            })
    df = pd.DataFrame(rows).drop_duplicates("bodyId")
    df.to_csv(out_csv, index=False)
    print(f"Saved soma metadata: {out_csv} | rows={len(df)}")
    return df

all_body_ids_for_somas = []
for ds in datasets.values():
    all_body_ids_for_somas.extend(ds["assignments"]["bodyId"].astype(int).tolist())

soma_metadata_df = fetch_neuprint_soma_metadata(all_body_ids_for_somas, label="all_loaded_datasets")

SOMA_LOOKUP = {}
if not soma_metadata_df.empty:
    for _, r in soma_metadata_df.iterrows():
        loc = np.array([r.get("soma_x", np.nan), r.get("soma_y", np.nan), r.get("soma_z", np.nan)], dtype=float)
        if np.isfinite(loc).all():
            SOMA_LOOKUP[int(r["bodyId"])] = {
                "xyz": loc,
                "radius": float(r.get("soma_radius", np.nan)) if pd.notna(r.get("soma_radius", np.nan)) else np.nan,
                "source": r.get("soma_source", "neuprint"),
            }
print(f"Soma lookup entries with finite coordinates: {len(SOMA_LOOKUP)}")


## Soma/cell-body rendering note

This plotting version separates the visual inspection into three views per cluster:

1. **Skeletons only** — old morphology skeleton traces without soma meshes.
2. **Somas only** — local SWC-radius soma/root meshes without projections.
3. **Combined** — skeleton traces + soma/root meshes.

This should make it much easier to judge whether the soma rendering helps or distracts, and whether projection patterns are consistent within a cluster. All cells in a cluster use the same stable cluster color across all three views.

If true per-body segmentation meshes are available locally, put files in `body_meshes/<bodyId>.obj` or `body_meshes/<bodyId>.ply` and use `SOMA_RENDER_MODE = "local_body_mesh"`.

In [ ]:
# ============================================================
# 7. 3D rendering helpers: shell + neurons + SWC-radius soma meshes
# ============================================================

if not HAS_PLOTLY:
    raise ImportError("Plotly is required for the interactive 3D cluster renderings.")


def rgba(hex_color, alpha=1.0):
    """Convert '#RRGGBB' to plotly rgba string."""
    hex_color = str(hex_color).lstrip("#")
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


def add_shell_trace(fig, alpha=SHELL_ALPHA):
    if not SHOW_BRAIN_SHELL:
        return fig
    if outer_shell_vertices_render is None or outer_shell_faces is None or len(outer_shell_vertices_render) == 0:
        return fig
    v = outer_shell_vertices_render
    f = outer_shell_faces
    fig.add_trace(go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        opacity=alpha,
        color="rgb(210, 215, 220)",
        name="outer brain shell",
        showscale=False,
        hoverinfo="skip",
        flatshading=False,
        lighting=dict(ambient=0.82, diffuse=0.35, specular=0.05, roughness=0.95),
    ))
    return fig


def soma_for_body(body_id, swc_dir):
    """Return soma center/radius/source using neuPrint metadata, then SWC fallback."""
    bid = int(body_id)
    if bid in SOMA_LOOKUP:
        d = SOMA_LOOKUP[bid]
        return d["xyz"], d.get("radius", np.nan), d.get("source", "neuprint")
    _, swc_soma = swc_segments_for_body(bid, swc_dir)
    if swc_soma is not None:
        return swc_soma, np.nan, "swc_fallback"
    return None, np.nan, "missing"


def sphere_mesh_vertices_faces(center, radius, theta_steps=9, phi_steps=7):
    """Return vertices/faces for a small sphere mesh."""
    center = np.asarray(center, dtype=float)
    radius = float(radius)
    theta = np.linspace(0, 2*np.pi, theta_steps, endpoint=False)
    phi = np.linspace(0, np.pi, phi_steps)
    verts = []
    for p in phi:
        for t in theta:
            verts.append([
                center[0] + radius * np.cos(t) * np.sin(p),
                center[1] + radius * np.sin(t) * np.sin(p),
                center[2] + radius * np.cos(p),
            ])
    faces = []
    for pi in range(phi_steps - 1):
        for ti in range(theta_steps):
            a = pi * theta_steps + ti
            b = pi * theta_steps + ((ti + 1) % theta_steps)
            c = (pi + 1) * theta_steps + ti
            d = (pi + 1) * theta_steps + ((ti + 1) % theta_steps)
            faces.append([a, c, b])
            faces.append([b, c, d])
    return np.asarray(verts, dtype=float), np.asarray(faces, dtype=int)


def cylinder_mesh_vertices_faces(a, b, radius, sides=9):
    """Return vertices/faces for a cylinder between two 3D points."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    axis = b - a
    length = np.linalg.norm(axis)
    if not np.isfinite(length) or length <= 1e-9:
        return np.empty((0, 3)), np.empty((0, 3), dtype=int)
    axis = axis / length
    # Choose an arbitrary perpendicular basis.
    tmp = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(tmp, axis)) > 0.9:
        tmp = np.array([0.0, 1.0, 0.0])
    u = np.cross(axis, tmp)
    u = u / np.linalg.norm(u)
    v = np.cross(axis, u)
    angles = np.linspace(0, 2*np.pi, sides, endpoint=False)
    ring_a, ring_b = [], []
    for t in angles:
        off = radius * (np.cos(t) * u + np.sin(t) * v)
        ring_a.append(a + off)
        ring_b.append(b + off)
    verts = np.vstack([ring_a, ring_b])
    faces = []
    for i in range(sides):
        j = (i + 1) % sides
        faces.append([i, j, sides + i])
        faces.append([j, sides + j, sides + i])
    return verts, np.asarray(faces, dtype=int)


def combine_mesh_parts(parts):
    """Combine list of (vertices, faces) mesh parts."""
    verts_all = []
    faces_all = []
    offset = 0
    for verts, faces in parts:
        if verts is None or faces is None or len(verts) == 0 or len(faces) == 0:
            continue
        verts_all.append(verts)
        faces_all.append(faces + offset)
        offset += len(verts)
    if not verts_all:
        return np.empty((0, 3)), np.empty((0, 3), dtype=int)
    return np.vstack(verts_all), np.vstack(faces_all)


def add_mesh3d(fig, vertices, faces, color, name, opacity=0.9, hoverinfo="skip", showlegend=False):
    if vertices is None or faces is None or len(vertices) == 0 or len(faces) == 0:
        return fig
    v = transform_points_for_render(vertices)
    fig.add_trace(go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color=color,
        opacity=opacity,
        name=name,
        showscale=False,
        hoverinfo=hoverinfo,
        flatshading=False,
        lighting=dict(ambient=0.45, diffuse=0.65, specular=0.2, roughness=0.75),
        showlegend=showlegend,
    ))
    return fig


def closest_swc_node_to_point(df, point):
    if df is None or df.empty or point is None:
        return None
    pts = df[["x", "y", "z"]].values.astype(float)
    point = np.asarray(point, dtype=float)
    d = np.linalg.norm(pts - point[None, :], axis=1)
    return int(df.iloc[int(np.nanargmin(d))]["node_id"])


def soma_anchor_node(df, soma_xyz=None):
    """Best node to anchor soma rendering."""
    if df is None or df.empty:
        return None
    if soma_xyz is not None and np.isfinite(np.asarray(soma_xyz, dtype=float)).all():
        n = closest_swc_node_to_point(df, soma_xyz)
        if n is not None:
            return n
    soma_nodes = df.loc[df["type"].astype(int) == 1]
    if len(soma_nodes):
        return int(soma_nodes.iloc[0]["node_id"])
    roots = df.loc[df["parent"].astype(int) == -1]
    if len(roots):
        return int(roots.iloc[0]["node_id"])
    if "radius" in df.columns and df["radius"].notna().any():
        return int(df.loc[df["radius"].astype(float).idxmax()]["node_id"])
    return int(df.iloc[0]["node_id"])


def local_soma_node_set(df, soma_xyz, anchor):
    """Choose compact local SWC node set around soma/root, emphasizing large radii."""
    df = df.copy()
    ids = set(df["node_id"].astype(int))
    coords = df.set_index("node_id")[["x", "y", "z"]]
    radii = df.set_index("node_id")["radius"].astype(float)
    adjacency = defaultdict(list)
    for p, n in swc_edges(df):
        if p in ids and n in ids:
            adjacency[int(p)].append(int(n))
            adjacency[int(n)].append(int(p))

    visited = {int(anchor): 0}
    frontier = [int(anchor)]
    for _ in range(SOMA_RADIUS_MESH_GRAPH_HOPS):
        new_frontier = []
        for u in frontier:
            for v in adjacency.get(u, []):
                if v not in visited:
                    visited[v] = visited[u] + 1
                    new_frontier.append(v)
        frontier = new_frontier
        if not frontier:
            break
    graph_nodes = set(visited.keys())

    if soma_xyz is None or not np.isfinite(np.asarray(soma_xyz, dtype=float)).all():
        soma_xyz = coords.loc[anchor].values.astype(float)
    soma_xyz = np.asarray(soma_xyz, dtype=float)
    all_pts = coords.values.astype(float)
    all_ids = coords.index.astype(int).to_numpy()
    d = np.linalg.norm(all_pts - soma_xyz[None, :], axis=1)
    euclidean_nodes = set(all_ids[d <= SOMA_RADIUS_MESH_EUCLIDEAN_RADIUS].astype(int))

    # Include local high-radius nodes; this tends to capture soma/root swelling better than distance alone.
    rr = radii.reindex(all_ids).values.astype(float)
    valid_rr = rr[np.isfinite(rr) & (rr > 0)]
    high_radius_nodes = set()
    if len(valid_rr):
        threshold = np.nanpercentile(valid_rr, 85)
        high_radius_nodes = set(all_ids[(rr >= threshold) & (d <= SOMA_RADIUS_MESH_EUCLIDEAN_RADIUS * 1.4)].astype(int))

    chosen = list((graph_nodes | euclidean_nodes | high_radius_nodes).intersection(ids))
    if len(chosen) < SOMA_RADIUS_MESH_MIN_NODES:
        nearest = all_ids[np.argsort(d)[:SOMA_RADIUS_MESH_MIN_NODES]].astype(int).tolist()
        chosen = list(set(chosen) | set(nearest))
    if len(chosen) > SOMA_RADIUS_MESH_MAX_NODES:
        chosen_arr = np.array(chosen, dtype=int)
        pts_chosen = coords.loc[chosen_arr].values.astype(float)
        dd = np.linalg.norm(pts_chosen - soma_xyz[None, :], axis=1)
        # Prefer nearby + high radius.
        rr_chosen = radii.loc[chosen_arr].values.astype(float)
        rr_norm = np.nan_to_num(rr_chosen / (np.nanmax(rr_chosen) if np.nanmax(rr_chosen) > 0 else 1), nan=0)
        score = dd - 0.5 * SOMA_RADIUS_MESH_EUCLIDEAN_RADIUS * rr_norm
        chosen = chosen_arr[np.argsort(score)[:SOMA_RADIUS_MESH_MAX_NODES]].astype(int).tolist()
    return sorted(set(map(int, chosen)))


def build_swc_radius_soma_mesh(body_id, swc_dir):
    """Build a compact soma/root mesh from local SWC nodes/radii."""
    path = swc_path_for_body(body_id, swc_dir)
    if path is None:
        return None
    df = read_swc(path)
    if df.empty:
        return None

    soma_xyz, soma_radius, soma_source = soma_for_body(body_id, swc_dir)
    anchor = soma_anchor_node(df, soma_xyz)
    if anchor is None:
        return None

    coords = df.set_index("node_id")[["x", "y", "z"]]
    radii = df.set_index("node_id")["radius"].astype(float)
    chosen = local_soma_node_set(df, soma_xyz, anchor)
    chosen_set = set(chosen)

    # Radii: use SWC radius, with conservative clipping to avoid ugly giant blobs.
    rr = radii.loc[chosen].values.astype(float)
    good = np.isfinite(rr) & (rr > 0)
    fallback_r = np.nanmedian(rr[good]) if np.any(good) else (soma_radius if np.isfinite(soma_radius) and soma_radius > 0 else 60.0)
    rr = np.where(good, rr, fallback_r)
    rr = np.clip(rr * SOMA_RADIUS_MESH_RADIUS_SCALE, SOMA_RADIUS_MESH_MIN_RADIUS, SOMA_RADIUS_MESH_MAX_RADIUS)

    parts = []
    # Sphere caps at selected local nodes.
    for nid, radius in zip(chosen, rr):
        center = coords.loc[nid].values.astype(float)
        parts.append(sphere_mesh_vertices_faces(
            center, radius,
            theta_steps=SOMA_RADIUS_MESH_SPHERE_THETA,
            phi_steps=SOMA_RADIUS_MESH_SPHERE_PHI,
        ))

    # Cylinders along local SWC edges. Radius is the smaller of endpoint radii.
    rr_by_id = dict(zip(chosen, rr))
    for p, n in swc_edges(df):
        if p in chosen_set and n in chosen_set:
            a = coords.loc[p].values.astype(float)
            b = coords.loc[n].values.astype(float)
            rad = min(rr_by_id.get(p, fallback_r), rr_by_id.get(n, fallback_r))
            parts.append(cylinder_mesh_vertices_faces(a, b, rad, sides=SOMA_RADIUS_MESH_CYLINDER_SIDES))

    vertices, faces = combine_mesh_parts(parts)
    if len(vertices) == 0 or len(faces) == 0:
        return None
    return {
        "vertices": vertices,
        "faces": faces,
        "source": soma_source,
        "n_nodes": len(chosen),
        "center": soma_xyz if soma_xyz is not None else coords.loc[anchor].values.astype(float),
    }


def read_obj_mesh(path):
    verts, faces = [], []
    with open(path, "r", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line.startswith("v "):
                parts = line.split()
                if len(parts) >= 4:
                    verts.append([float(parts[1]), float(parts[2]), float(parts[3])])
            elif line.startswith("f "):
                inds = []
                for p in line.split()[1:]:
                    try:
                        inds.append(int(p.split("/")[0]) - 1)
                    except Exception:
                        pass
                if len(inds) >= 3:
                    # Triangulate fan if needed.
                    for i in range(1, len(inds)-1):
                        faces.append([inds[0], inds[i], inds[i+1]])
    return np.asarray(verts, dtype=float), np.asarray(faces, dtype=int)


def read_ply_ascii_mesh(path):
    """Minimal ASCII PLY reader. Binary PLY is not supported here."""
    with open(path, "r", errors="ignore") as f:
        lines = f.readlines()
    if not lines or not lines[0].startswith("ply"):
        raise ValueError("Not a PLY file")
    n_vertices = n_faces = None
    end_idx = None
    for i, line in enumerate(lines):
        if line.startswith("format") and "ascii" not in line:
            raise ValueError("Only ASCII PLY supported by this lightweight reader")
        if line.startswith("element vertex"):
            n_vertices = int(line.split()[-1])
        if line.startswith("element face"):
            n_faces = int(line.split()[-1])
        if line.strip() == "end_header":
            end_idx = i + 1
            break
    if end_idx is None or n_vertices is None:
        raise ValueError("Malformed PLY")
    verts = []
    for line in lines[end_idx:end_idx+n_vertices]:
        parts = line.split()
        verts.append([float(parts[0]), float(parts[1]), float(parts[2])])
    faces = []
    face_start = end_idx + n_vertices
    for line in lines[face_start:face_start+(n_faces or 0)]:
        vals = [int(x) for x in line.split()]
        k = vals[0]
        inds = vals[1:1+k]
        for i in range(1, len(inds)-1):
            faces.append([inds[0], inds[i], inds[i+1]])
    return np.asarray(verts, dtype=float), np.asarray(faces, dtype=int)


def local_body_mesh_path(body_id):
    if not LOCAL_BODY_MESH_DIR.exists():
        return None
    for ext in [".obj", ".ply"]:
        p = LOCAL_BODY_MESH_DIR / f"{int(body_id)}{ext}"
        if p.exists():
            return p
    matches = []
    for ext in ["*.obj", "*.ply"]:
        matches.extend(LOCAL_BODY_MESH_DIR.glob(f"*{int(body_id)}*{ext[-4:]}"))
    return matches[0] if matches else None


def clipped_local_body_soma_mesh(body_id, swc_dir):
    """If local true body mesh exists, clip soma neighborhood and return mesh."""
    path = local_body_mesh_path(body_id)
    if path is None:
        return None
    soma_xyz, soma_radius, source = soma_for_body(body_id, swc_dir)
    if soma_xyz is None:
        return None
    try:
        if str(path).lower().endswith(".obj"):
            verts, faces = read_obj_mesh(path)
        else:
            verts, faces = read_ply_ascii_mesh(path)
    except Exception as e:
        print(f"Could not read local body mesh for {body_id}: {e}")
        return None
    if len(verts) == 0 or len(faces) == 0:
        return None
    d = np.linalg.norm(verts - np.asarray(soma_xyz)[None, :], axis=1)
    keep_v = d <= LOCAL_BODY_MESH_SOMA_CLIP_RADIUS
    keep_faces = keep_v[faces].all(axis=1)
    faces2 = faces[keep_faces]
    if len(faces2) == 0:
        return None
    # Limit if too large.
    if len(faces2) > LOCAL_BODY_MESH_MAX_FACES:
        rng = np.random.default_rng(42)
        faces2 = faces2[np.sort(rng.choice(len(faces2), LOCAL_BODY_MESH_MAX_FACES, replace=False))]
    used = np.unique(faces2.ravel())
    remap = {old: new for new, old in enumerate(used)}
    verts_new = verts[used]
    faces_new = np.vectorize(remap.get)(faces2)
    return {"vertices": verts_new, "faces": faces_new.astype(int), "source": f"local_body_mesh:{path.name}"}


def add_soma_trace(fig, body_id, swc_dir, color, show_text=False):
    if SOMA_RENDER_MODE == "local_body_mesh" or (PREFER_LOCAL_BODY_MESH_SOMA and SOMA_RENDER_MODE == "swc_radius_mesh"):
        mesh = clipped_local_body_soma_mesh(body_id, swc_dir)
        if mesh is not None:
            fig = add_mesh3d(
                fig, mesh["vertices"], mesh["faces"],
                color=color,
                name=f"true/local soma mesh {int(body_id)}",
                opacity=LOCAL_BODY_MESH_OPACITY,
                hoverinfo="text",
                showlegend=False,
            )
            return fig, True

    if SOMA_RENDER_MODE == "swc_radius_mesh":
        mesh = build_swc_radius_soma_mesh(body_id, swc_dir)
        if mesh is not None:
            fig = add_mesh3d(
                fig, mesh["vertices"], mesh["faces"],
                color=color,
                name=f"SWC-radius soma mesh {int(body_id)}",
                opacity=SOMA_RADIUS_MESH_OPACITY,
                hoverinfo="skip",
                showlegend=False,
            )
            # Optional label at mesh center.
            if show_text:
                c = transform_points_for_render(mesh["center"])
                fig.add_trace(go.Scatter3d(
                    x=[c[0]], y=[c[1]], z=[c[2]], mode="text",
                    text=[str(int(body_id))], textposition="top center",
                    showlegend=False, hoverinfo="skip",
                ))
            return fig, str(mesh.get("source", "")).startswith("neuprint")

    # Simple marker/sphere fallback.
    soma, radius, source = soma_for_body(body_id, swc_dir)
    if soma is None:
        return fig, False
    soma_r = transform_points_for_render(soma)
    label = f"soma {int(body_id)} ({source})"
    if SOMA_RENDER_MODE == "sphere" and np.isfinite(radius) and radius > 0:
        verts, faces = sphere_mesh_vertices_faces(soma, radius * SOMA_SPHERE_RADIUS_SCALE, theta_steps=10, phi_steps=8)
        fig = add_mesh3d(fig, verts, faces, color=color, name=label, opacity=SOMA_SPHERE_OPACITY)
    else:
        size = SOMA_MARKER_SIZE
        if np.isfinite(radius) and radius > 0:
            size = int(np.clip(radius / 50, SOMA_MARKER_SIZE_MIN, SOMA_MARKER_SIZE_MAX))
        fig.add_trace(go.Scatter3d(
            x=[soma_r[0]], y=[soma_r[1]], z=[soma_r[2]],
            mode="markers+text" if show_text else "markers",
            marker=dict(size=size, symbol="circle", color=color),
            text=[str(int(body_id))] if show_text else None,
            textposition="top center",
            name=label,
            hovertext=label,
            hoverinfo="text",
            showlegend=False,
        ))
    return fig, source.startswith("neuprint")


def build_neuron_radius_tube_mesh(body_id, swc_dir):
    """Experimental full-skeleton tube mesh from SWC radii. Heavy but tunable."""
    path = swc_path_for_body(body_id, swc_dir)
    if path is None:
        return None
    df = read_swc(path)
    if df.empty:
        return None
    coords = df.set_index("node_id")[["x", "y", "z"]]
    radii = df.set_index("node_id")["radius"].astype(float)
    parts = []
    edges = swc_edges(df)[:CELL_TUBE_MAX_EDGES_PER_NEURON]
    for p, n in edges:
        try:
            a = coords.loc[p].values.astype(float)
            b = coords.loc[n].values.astype(float)
            r = np.nanmean([radii.loc[p], radii.loc[n]])
            if not np.isfinite(r) or r <= 0:
                r = CELL_TUBE_MIN_RADIUS
            r = np.clip(r * CELL_TUBE_RADIUS_SCALE, CELL_TUBE_MIN_RADIUS, CELL_TUBE_MAX_RADIUS)
            parts.append(cylinder_mesh_vertices_faces(a, b, r, sides=CELL_TUBE_SIDES))
        except Exception:
            pass
    verts, faces = combine_mesh_parts(parts)
    if len(verts) == 0:
        return None
    return {"vertices": verts, "faces": faces}


def add_single_neuron_trace(fig, body_id, swc_dir, cluster_color, width=CELL_LINE_WIDTH):
    if CELL_RENDER_MODE == "radius_tubes":
        mesh = build_neuron_radius_tube_mesh(body_id, swc_dir)
        if mesh is not None:
            return add_mesh3d(
                fig, mesh["vertices"], mesh["faces"],
                color=rgba(cluster_color, CELL_LINE_ALPHA),
                name=f"neuron radius tube {int(body_id)}",
                opacity=CELL_LINE_ALPHA,
                hoverinfo="skip",
                showlegend=False,
            )
    # Fast clean line rendering fallback/default.
    segs, _ = swc_segments_for_body(body_id, swc_dir)
    if not segs:
        return fig
    xs, ys, zs = [], [], []
    for a, b in segs:
        aa = transform_points_for_render(a)
        bb = transform_points_for_render(b)
        xs += [aa[0], bb[0], None]
        ys += [aa[1], bb[1], None]
        zs += [aa[2], bb[2], None]
    fig.add_trace(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode="lines",
        line=dict(width=width, color=rgba(cluster_color, CELL_LINE_ALPHA)),
        name=f"cluster neuron {int(body_id)}",
        hovertext=str(int(body_id)),
        hoverinfo="text",
        showlegend=False,
    ))
    return fig


def add_neuron_traces(fig, body_ids, swc_dir, cluster_color, width=CELL_LINE_WIDTH, show_somas=True, show_soma_text=False):
    n_soma_neuprint = 0
    for bid in body_ids:
        fig = add_single_neuron_trace(fig, bid, swc_dir, cluster_color=cluster_color, width=width)
        if show_somas:
            fig, is_neuprint = add_soma_trace(fig, bid, swc_dir, color=cluster_color, show_text=show_soma_text)
            n_soma_neuprint += int(is_neuprint)
    return fig, n_soma_neuprint


def add_population_context(fig, ds, max_cells=MAX_CONTEXT_CELLS):
    a = ds["assignments"]
    ids = a["bodyId"].astype(int).tolist()[:max_cells]
    swc_dir = ds["raw_swc_dir"]
    for bid in ids:
        segs, _ = swc_segments_for_body(bid, swc_dir)
        if not segs:
            continue
        xs, ys, zs = [], [], []
        for p, q in segs:
            pp = transform_points_for_render(p)
            qq = transform_points_for_render(q)
            xs += [pp[0], qq[0], None]
            ys += [pp[1], qq[1], None]
            zs += [pp[2], qq[2], None]
        fig.add_trace(go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode="lines",
            line=dict(width=CONTEXT_LINE_WIDTH, color=f"rgba(0,0,0,{POPULATION_CONTEXT_ALPHA})"),
            showlegend=False,
            hoverinfo="skip",
        ))
    return fig


def render_cluster_3d(ds, dataset_key, cluster, save_html=True, show_population_context=False, show_somas=True, show_soma_text=False):
    a = ds["assignments"]
    cluster = int(cluster)
    body_ids = a.loc[a["cluster"] == cluster, "bodyId"].astype(int).tolist()
    swc_dir = ds["raw_swc_dir"]
    ccolor = get_cluster_color(dataset_key, cluster)

    fig = go.Figure()
    add_shell_trace(fig, alpha=SHELL_ALPHA)
    if show_population_context:
        add_population_context(fig, ds)
    body_ids_to_draw = body_ids[:MAX_CELLS_PER_CLUSTER_RENDER]
    fig, n_neuprint_somas = add_neuron_traces(
        fig,
        body_ids_to_draw,
        swc_dir,
        cluster_color=ccolor,
        width=CELL_LINE_WIDTH,
        show_somas=show_somas,
        show_soma_text=show_soma_text,
    )

    id_text = ", ".join(map(str, body_ids[:35])) + ("..." if len(body_ids) > 35 else "")
    title = (
        f"{ds['label']} | cluster {cluster} | n={len(body_ids)}"
        f"<br>cluster color: {ccolor} | skeleton mode: {CELL_RENDER_MODE} | line width: {CELL_LINE_WIDTH} | soma mode: {SOMA_RENDER_MODE}"
        f"<br>neuPrint soma centers used: {n_neuprint_somas}/{len(body_ids_to_draw)} | IDs: {id_text}"
    )
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="x", yaxis_title="y", zaxis_title="z",
            aspectmode="data",
            xaxis=dict(showbackground=False),
            yaxis=dict(showbackground=False),
            zaxis=dict(showbackground=False),
        ),
        width=1050,
        height=850,
        legend=dict(itemsizing="constant"),
        margin=dict(l=0, r=0, t=110, b=0),
    )
    if save_html and SAVE_INTERACTIVE_HTML:
        outdir = RENDER_DIR / dataset_key / "interactive_3d_v17_radius_soma_meshes"
        outdir.mkdir(parents=True, exist_ok=True)
        html = outdir / f"{dataset_key}_cluster_{cluster:02d}_v17_radiusSomaMesh.html"
        fig.write_html(str(html))
        print(f"Saved HTML: {html}")
    return fig


def inspect_cluster(dataset_key="selected_raphe", cluster=1, show_population_context=False, show_somas=True, show_soma_text=False):
    ds = datasets[dataset_key]
    ids = ds["assignments"].loc[ds["assignments"]["cluster"] == int(cluster), "bodyId"].astype(int).tolist()
    print(f"{ds['label']} | cluster {int(cluster)} | n={len(ids)} | color={get_cluster_color(dataset_key, cluster)}")
    print("IDs:", ", ".join(map(str, ids)))
    display(pd.DataFrame({"bodyId": ids}))
    fig = render_cluster_3d(
        ds, dataset_key, int(cluster),
        save_html=True,
        show_population_context=show_population_context,
        show_somas=show_somas,
        show_soma_text=show_soma_text,
    )
    display(fig)
    return fig


# ============================================================
# v18 overrides: render skeleton-only, soma-only, or combined views
# ============================================================

RENDER_MODE_LABELS = {
    "skeletons_only": "Skeletons only",
    "somas_only": "Somas only",
    "combined": "Skeletons + somas",
}


def _safe_cluster_ids(ds, cluster):
    a = ds["assignments"]
    cluster = int(cluster)
    return a.loc[a["cluster"] == cluster, "bodyId"].astype(int).tolist()


def render_cluster_3d_mode(
    ds,
    dataset_key,
    cluster,
    render_mode="combined",
    save_html=True,
    show_population_context=False,
    show_soma_text=False,
):
    """
    Render one cluster in one of three modes:
      - skeletons_only: only the old skeleton traces
      - somas_only: only soma/root radius meshes
      - combined: skeleton traces + soma/root meshes
    """
    if render_mode not in RENDER_MODE_LABELS:
        raise ValueError(f"render_mode must be one of {list(RENDER_MODE_LABELS)}")

    cluster = int(cluster)
    body_ids = _safe_cluster_ids(ds, cluster)
    body_ids_to_draw = body_ids[:MAX_CELLS_PER_CLUSTER_RENDER]
    swc_dir = ds["raw_swc_dir"]
    ccolor = get_cluster_color(dataset_key, cluster)

    fig = go.Figure()
    add_shell_trace(fig, alpha=SHELL_ALPHA)

    # Context is usually most useful in skeleton/combined mode; allow it everywhere if requested.
    if show_population_context:
        add_population_context(fig, ds)

    n_somas = 0
    for bid in body_ids_to_draw:
        if render_mode in ["skeletons_only", "combined"]:
            fig = add_single_neuron_trace(
                fig,
                bid,
                swc_dir,
                cluster_color=ccolor,
                width=CELL_LINE_WIDTH,
            )
        if render_mode in ["somas_only", "combined"]:
            fig, is_neuprint = add_soma_trace(
                fig,
                bid,
                swc_dir,
                color=ccolor,
                show_text=show_soma_text,
            )
            n_somas += int(is_neuprint)

    id_text = ", ".join(map(str, body_ids[:35])) + ("..." if len(body_ids) > 35 else "")
    title = (
        f"{ds['label']} | cluster {cluster} | n={len(body_ids)} | {RENDER_MODE_LABELS[render_mode]}"
        f"<br>cluster color: {ccolor} | line width: {CELL_LINE_WIDTH} | soma mode: {SOMA_RENDER_MODE}"
        f"<br>somas drawn: {len(body_ids_to_draw) if render_mode in ['somas_only', 'combined'] else 0}; "
        f"neuPrint soma anchors: {n_somas}/{len(body_ids_to_draw)} | IDs: {id_text}"
    )

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="x", yaxis_title="y", zaxis_title="z",
            aspectmode="data",
            xaxis=dict(showbackground=False),
            yaxis=dict(showbackground=False),
            zaxis=dict(showbackground=False),
        ),
        width=1050,
        height=850,
        legend=dict(itemsizing="constant"),
        margin=dict(l=0, r=0, t=120, b=0),
    )

    if save_html and SAVE_INTERACTIVE_HTML and SAVE_EACH_RENDER_MODE_HTML:
        outdir = RENDER_DIR / dataset_key / "interactive_3d_v18_three_brain_views"
        outdir.mkdir(parents=True, exist_ok=True)
        html = outdir / f"{dataset_key}_cluster_{cluster:02d}_{render_mode}_v18.html"
        fig.write_html(str(html))
        print(f"Saved HTML: {html}")

    return fig


def render_cluster_three_views(
    ds,
    dataset_key,
    cluster,
    save_html=True,
    show_population_context=False,
    show_soma_text=False,
):
    """Render the three requested views as three separate interactive Plotly brains."""
    figs = {}
    for mode in ["skeletons_only", "somas_only", "combined"]:
        print(f"Rendering {RENDER_MODE_LABELS[mode]}...")
        figs[mode] = render_cluster_3d_mode(
            ds,
            dataset_key,
            cluster,
            render_mode=mode,
            save_html=save_html,
            show_population_context=show_population_context,
            show_soma_text=show_soma_text,
        )
    return figs


# Backward-compatible wrapper. Existing calls still work, but default to combined.
def render_cluster_3d(
    ds,
    dataset_key,
    cluster,
    save_html=True,
    show_population_context=False,
    show_somas=True,
    show_soma_text=False,
    render_mode="combined",
):
    if not show_somas and render_mode == "combined":
        render_mode = "skeletons_only"
    return render_cluster_3d_mode(
        ds,
        dataset_key,
        cluster,
        render_mode=render_mode,
        save_html=save_html,
        show_population_context=show_population_context,
        show_soma_text=show_soma_text,
    )


def inspect_cluster(
    dataset_key="selected_raphe",
    cluster=1,
    show_population_context=False,
    show_somas=True,
    show_soma_text=False,
    render_mode=DEFAULT_CLUSTER_RENDER_MODE,
):
    """
    Inspect a cluster. By default, render_mode='all_three' shows:
    skeletons-only, somas-only, and combined.
    """
    ds = datasets[dataset_key]
    ids = _safe_cluster_ids(ds, cluster)
    print(f"{ds['label']} | cluster {int(cluster)} | n={len(ids)} | color={get_cluster_color(dataset_key, cluster)}")
    print("IDs:", ", ".join(map(str, ids)))
    display(pd.DataFrame({"bodyId": ids}))

    if render_mode == "all_three":
        figs = render_cluster_three_views(
            ds,
            dataset_key,
            int(cluster),
            save_html=True,
            show_population_context=show_population_context,
            show_soma_text=show_soma_text,
        )
        for mode, fig in figs.items():
            display(HTML(f"<h3>{RENDER_MODE_LABELS[mode]}</h3>"))
            display(fig)
        return figs

    fig = render_cluster_3d_mode(
        ds,
        dataset_key,
        int(cluster),
        render_mode=render_mode,
        save_html=True,
        show_population_context=show_population_context,
        show_soma_text=show_soma_text,
    )
    display(fig)
    return fig


In [ ]:
# ============================================================
# 8. Interactive cluster selector: three separate brain views
# ============================================================

def launch_cluster_selector():
    if not HAS_WIDGETS:
        print("ipywidgets not available. Use inspect_cluster('selected_raphe', 1, render_mode='all_three').")
        return

    dataset_options = [(ds["label"], key) for key, ds in datasets.items()]
    dataset_dd = widgets.Dropdown(options=dataset_options, description="Dataset:", layout=widgets.Layout(width="420px"))
    cluster_dd = widgets.Dropdown(description="Cluster:", layout=widgets.Layout(width="260px"))
    render_mode_dd = widgets.Dropdown(
        options=[
            ("All three: skeletons, somas, combined", "all_three"),
            ("Skeletons only", "skeletons_only"),
            ("Somas only", "somas_only"),
            ("Combined", "combined"),
        ],
        value=DEFAULT_CLUSTER_RENDER_MODE,
        description="View:",
        layout=widgets.Layout(width="420px"),
    )
    show_context = widgets.Checkbox(value=False, description="Population context")
    show_soma_text = widgets.Checkbox(value=False, description="Soma ID labels")
    out = widgets.Output()

    def update_clusters(*args):
        key = dataset_dd.value
        ds = datasets[key]
        sizes = ds["assignments"]["cluster"].value_counts().sort_index()
        opts = [(f"C{int(c):02d} | n={int(s)}", int(c)) for c, s in sizes.items()]
        cluster_dd.options = opts

    def draw(*args):
        with out:
            clear_output(wait=True)
            key = dataset_dd.value
            cl = cluster_dd.value
            if cl is None:
                return
            inspect_cluster(
                key,
                cl,
                show_population_context=show_context.value,
                show_soma_text=show_soma_text.value,
                render_mode=render_mode_dd.value,
            )

    dataset_dd.observe(update_clusters, names="value")
    cluster_dd.observe(draw, names="value")
    render_mode_dd.observe(draw, names="value")
    show_context.observe(draw, names="value")
    show_soma_text.observe(draw, names="value")

    update_clusters()
    display(widgets.VBox([
        widgets.HTML("<b>Interactive cluster renderer — choose skeletons-only, somas-only, combined, or all three</b>"),
        widgets.HBox([dataset_dd, cluster_dd]),
        widgets.HBox([render_mode_dd]),
        widgets.HBox([show_context, show_soma_text]),
        out,
    ]))
    draw()

launch_cluster_selector()


In [ ]:
# ============================================================
# 9. Optional: batch-generate HTML renders for all non-singleton clusters
# ============================================================

# Set to True and rerun this cell if you want to regenerate HTML for every non-singleton cluster.
BATCH_RENDER_ALL_NON_SINGLETON_CLUSTERS = False

# Choose which views to batch-save.
# Options: ["skeletons_only", "somas_only", "combined"]
BATCH_RENDER_MODES = ["skeletons_only", "somas_only", "combined"]

if BATCH_RENDER_ALL_NON_SINGLETON_CLUSTERS:
    for key, ds in datasets.items():
        sizes = ds["assignments"]["cluster"].value_counts().sort_values(ascending=False)
        clusters = [int(c) for c, s in sizes.items() if int(s) >= 2]
        print(f"{key}: rendering {len(clusters)} non-singleton clusters")
        for cl in tqdm(clusters, desc=f"Render {key}"):
            for mode in BATCH_RENDER_MODES:
                _ = render_cluster_3d_mode(
                    ds,
                    key,
                    cl,
                    render_mode=mode,
                    save_html=True,
                    show_population_context=False,
                    show_soma_text=False,
                )
else:
    print("Batch rendering disabled. Set BATCH_RENDER_ALL_NON_SINGLETON_CLUSTERS = True to save all cluster HTML renders.")


In [ ]:
# ============================================================
# Add-on for v19: selected raphe neurons in yellow + brain shell
# ============================================================
# Paste this near the end of v19 after the SWC/shell/rendering helpers.
# It plots the ~234 selected raphe neurons in yellow inside the translucent
# brain shell, with top / side / angled fixed-camera views and PDF/PNG/SVG export.

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import display

# ----------------------------
# User settings
# ----------------------------

SELECTED_RAPHE_DATASET_KEY = "selected_raphe"
SELECTED_RAPHE_FIG_DIR = FIG_DIR / "selected_raphe_all_neurons_yellow_3d"
SELECTED_RAPHE_FIG_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_RAPHE_COLOR = "rgb(255, 220, 0)"  # yellow
SELECTED_RAPHE_LINE_WIDTH = 2.0
SELECTED_RAPHE_LINE_ALPHA = 0.82

SELECTED_RAPHE_SHOW_SKELETONS = False
SELECTED_RAPHE_SHOW_SOMAS = True
SELECTED_RAPHE_SOMA_SIZE = 3.6
SELECTED_RAPHE_SOMA_ALPHA = 0.92

BRAIN_SHELL_COLOR = "rgb(180, 190, 205)"
BRAIN_SHELL_ALPHA_FOR_SELECTED_RAPHE = 0.13

SELECTED_RAPHE_STATIC_WIDTH = 1400
SELECTED_RAPHE_STATIC_HEIGHT = 1100
SELECTED_RAPHE_STATIC_SCALE = 2
SAVE_SELECTED_RAPHE_STATIC = True

# Set to None to plot all selected raphe IDs loaded in v19.
# Or set e.g. 80 for a quick test.
MAX_SELECTED_RAPHE_CELLS_TO_PLOT = None


# ----------------------------
# Helpers
# ----------------------------

def _require_v19_objects_for_selected_raphe_plot():
    required = [
        "datasets",
        "FIG_DIR",
        "outer_shell_vertices_render",
        "outer_shell_faces",
        "transform_points_for_render",
        "swc_segments_for_body",
        "soma_for_body",
    ]
    missing = [x for x in required if x not in globals()]
    if missing:
        raise RuntimeError(
            "Missing v19 variables/functions: "
            + ", ".join(missing)
            + ". Run the earlier v19 setup/shell/SWC cells first."
        )
    if SELECTED_RAPHE_DATASET_KEY not in datasets:
        raise RuntimeError(
            f"Dataset {SELECTED_RAPHE_DATASET_KEY!r} not found. "
            f"Available datasets: {list(datasets.keys())}"
        )


def _selected_raphe_body_ids():
    ds = datasets[SELECTED_RAPHE_DATASET_KEY]
    if "assignments" in ds and isinstance(ds["assignments"], pd.DataFrame):
        ids = ds["assignments"]["bodyId"].dropna().astype(int).tolist()
    elif "body_ids" in ds:
        ids = [int(x) for x in ds["body_ids"]]
    else:
        raise RuntimeError("Could not find selected raphe body IDs in v19 datasets dict.")
    ids = list(dict.fromkeys(ids))
    if MAX_SELECTED_RAPHE_CELLS_TO_PLOT is not None:
        ids = ids[:int(MAX_SELECTED_RAPHE_CELLS_TO_PLOT)]
    return ids


def _mesh3d_trace_selected(vertices, faces, name, color, opacity):
    vertices = np.asarray(vertices, dtype=float)
    faces = np.asarray(faces, dtype=int)
    return go.Mesh3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        name=name,
        color=color,
        opacity=opacity,
        flatshading=True,
        lighting=dict(
            ambient=0.68,
            diffuse=0.65,
            specular=0.12,
            roughness=0.8,
            fresnel=0.05,
        ),
        # Safe for older Plotly versions: coordinate must be <= 100000.
        lightposition=dict(x=0, y=0, z=100000),
        hoverinfo="name",
        showscale=False,
    )


def _camera_for_selected_raphe_view(view):
    view = str(view).lower()
    if view == "top":
        return dict(eye=dict(x=0.0, y=0.0, z=2.35), up=dict(x=0, y=1, z=0))
    if view == "side":
        return dict(eye=dict(x=2.35, y=0.0, z=0.0), up=dict(x=0, y=0, z=1))
    if view == "angled":
        return dict(eye=dict(x=1.55, y=1.65, z=1.15), up=dict(x=0, y=0, z=1))
    raise ValueError(f"Unknown view: {view}")


def _add_selected_raphe_skeletons_and_somas(fig, body_ids, swc_dir):
    all_x, all_y, all_z = [], [], []
    soma_xyz = []
    soma_text = []
    missing_swc = []

    for bid in body_ids:
        segs, _ = swc_segments_for_body(int(bid), swc_dir)
        if not segs:
            missing_swc.append(int(bid))
        elif SELECTED_RAPHE_SHOW_SKELETONS:
            for a, b in segs:
                pts = transform_points_for_render(np.vstack([a, b]))
                all_x += [pts[0, 0], pts[1, 0], None]
                all_y += [pts[0, 1], pts[1, 1], None]
                all_z += [pts[0, 2], pts[1, 2], None]

        if SELECTED_RAPHE_SHOW_SOMAS:
            center, radius, source = soma_for_body(int(bid), swc_dir)
            if center is not None:
                cxyz = transform_points_for_render(np.asarray(center, dtype=float).reshape(1, 3))[0]
                soma_xyz.append(cxyz)
                soma_text.append(f"bodyId: {int(bid)}<br>soma source: {source}")

    if SELECTED_RAPHE_SHOW_SKELETONS and len(all_x):
        fig.add_trace(
            go.Scatter3d(
                x=all_x,
                y=all_y,
                z=all_z,
                mode="lines",
                line=dict(width=SELECTED_RAPHE_LINE_WIDTH, color=SELECTED_RAPHE_COLOR),
                opacity=SELECTED_RAPHE_LINE_ALPHA,
                name=f"Selected raphe skeletons, n={len(body_ids)}",
                hoverinfo="skip",
            )
        )

    if SELECTED_RAPHE_SHOW_SOMAS and len(soma_xyz):
        soma_xyz = np.asarray(soma_xyz, dtype=float)
        fig.add_trace(
            go.Scatter3d(
                x=soma_xyz[:, 0],
                y=soma_xyz[:, 1],
                z=soma_xyz[:, 2],
                mode="markers",
                marker=dict(
                    size=SELECTED_RAPHE_SOMA_SIZE,
                    color=SELECTED_RAPHE_COLOR,
                    opacity=SELECTED_RAPHE_SOMA_ALPHA,
                ),
                name=f"Selected raphe soma/root points, n={len(soma_xyz)}",
                text=soma_text,
                hoverinfo="text",
            )
        )

    if missing_swc:
        print(f"Missing/no usable SWCs for {len(missing_swc)} cells. First few:", missing_swc[:20])

    return fig


def plot_selected_raphe_all_neurons_yellow_brain_shell_v19():
    _require_v19_objects_for_selected_raphe_plot()

    ds = datasets[SELECTED_RAPHE_DATASET_KEY]
    body_ids = _selected_raphe_body_ids()
    swc_dir = ds["raw_swc_dir"]

    print(f"Plotting selected raphe neurons: {len(body_ids)} cells")
    print("SWC dir:", swc_dir)

    brain_vertices_plot = np.asarray(outer_shell_vertices_render, dtype=float)
    brain_faces_plot = np.asarray(outer_shell_faces, dtype=int)

    views = ["top", "side", "angled"]
    figs = {}

    for view in views:
        fig = go.Figure()

        fig.add_trace(
            _mesh3d_trace_selected(
                brain_vertices_plot,
                brain_faces_plot,
                name="Outer brain shell",
                color=BRAIN_SHELL_COLOR,
                opacity=BRAIN_SHELL_ALPHA_FOR_SELECTED_RAPHE,
            )
        )

        fig = _add_selected_raphe_skeletons_and_somas(fig, body_ids, swc_dir)

        fig.update_layout(
            title=dict(
                text=f"Selected raphe neurons, n={len(body_ids)} — {view} view",
                x=0.5,
                xanchor="center",
            ),
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                aspectmode="data",
                camera=_camera_for_selected_raphe_view(view),
                bgcolor="white",
            ),
            paper_bgcolor="white",
            plot_bgcolor="white",
            margin=dict(l=0, r=0, t=55, b=0),
            width=SELECTED_RAPHE_STATIC_WIDTH,
            height=SELECTED_RAPHE_STATIC_HEIGHT,
            showlegend=True,
            legend=dict(
                x=0.02,
                y=0.98,
                bgcolor="rgba(255,255,255,0.72)",
            ),
        )

        display(fig)

        out_base = SELECTED_RAPHE_FIG_DIR / f"selected_raphe_all_neurons_yellow_{view}"

        if SAVE_INTERACTIVE_HTML:
            html_path = out_base.with_suffix(".html")
            fig.write_html(str(html_path))
            print(f"Saved HTML: {html_path}")

        if SAVE_SELECTED_RAPHE_STATIC:
            try:
                if SAVE_PNG:
                    png_path = out_base.with_suffix(".png")
                    fig.write_image(str(png_path), scale=SELECTED_RAPHE_STATIC_SCALE)
                    print(f"Saved PNG: {png_path}")
                if SAVE_PDF:
                    pdf_path = out_base.with_suffix(".pdf")
                    fig.write_image(str(pdf_path), scale=SELECTED_RAPHE_STATIC_SCALE)
                    print(f"Saved PDF: {pdf_path}")
                if SAVE_SVG:
                    svg_path = out_base.with_suffix(".svg")
                    fig.write_image(str(svg_path), scale=SELECTED_RAPHE_STATIC_SCALE)
                    print(f"Saved SVG: {svg_path}")
            except Exception as e:
                print(
                    "Static export failed. Interactive HTML was still saved if enabled.\n"
                    "To enable static export, install/update kaleido:\n"
                    "  pip install -U kaleido\n"
                    f"Error: {repr(e)}"
                )

        figs[view] = fig

    # Save selected body IDs next to the figures.
    ids_csv = SELECTED_RAPHE_FIG_DIR / "selected_raphe_plotted_bodyIds.csv"
    pd.DataFrame({"bodyId": body_ids}).to_csv(ids_csv, index=False)
    print("Saved plotted body ID list:", ids_csv)

    return figs, body_ids


selected_raphe_yellow_figs, selected_raphe_yellow_body_ids = (
    plot_selected_raphe_all_neurons_yellow_brain_shell_v19()
)

In [ ]:
# ============================================================
# 10. Check selected-raphe cluster membership + show IDs on UMAP
# ============================================================
# Edit CELLS_TO_CHECK and rerun this cell.
# This cell uses ONLY the selected_raphe embedding/clustering.

CELLS_TO_CHECK = [
    100791363,
    102967931,
    106511646,
    105165440,
    100427087,
    100312010,
    103164598,
    103159060,
    101079563,
    103160069,
    100098958,
    100668438,
    100528051,
    100463138,
    109988161,
    100098958,
    100463138,
]

SELECTED_RAPHE_DATASET_KEY = "selected_raphe"
CHECKED_CELL_FIG_DIR = PLOT_OUT_ROOT / "checked_cells_selected_raphe_umap"
CHECKED_CELL_FIG_DIR.mkdir(parents=True, exist_ok=True)

CHECK_UMAP_LABEL_IDS = True
CHECK_UMAP_POINT_SIZE = 45
CHECK_UMAP_HIGHLIGHT_SIZE = 150
CHECK_UMAP_HIGHLIGHT_COLOR = "yellow"
CHECK_UMAP_HIGHLIGHT_EDGE_COLOR = "black"


def selected_raphe_assignments():
    if SELECTED_RAPHE_DATASET_KEY not in datasets:
        raise KeyError(
            f"Dataset {SELECTED_RAPHE_DATASET_KEY!r} not loaded. "
            f"Available datasets: {list(datasets.keys())}"
        )
    ds = datasets[SELECTED_RAPHE_DATASET_KEY]
    a = ds["assignments"].copy()
    a["bodyId"] = a["bodyId"].astype(int)
    a["cluster"] = a["cluster"].astype(int)
    if "cluster_size" not in a.columns:
        sizes = a["cluster"].value_counts()
        a["cluster_size"] = a["cluster"].map(sizes).astype(int)
    if "cluster_name" not in a.columns:
        a["cluster_name"] = a["cluster"].map(lambda x: f"selected_raphe_C{int(x):02d}")
    return ds, a


def get_selected_raphe_embedding(method="umap"):
    """Return selected-raphe embedding DataFrame for method='umap' or 'mds'."""
    ds, a = selected_raphe_assignments()
    if "embeddings" not in ds:
        raise RuntimeError("No embeddings loaded for selected_raphe.")

    emb_obj = ds["embeddings"]
    if isinstance(emb_obj, dict):
        emb = emb_obj.get(method)
        if emb is None:
            raise RuntimeError(f"No {method} embedding found in ds['embeddings'] dict.")
        emb = emb.copy()
    else:
        emb = emb_obj.copy()

    emb["bodyId"] = emb["bodyId"].astype(int)
    possible_x = [f"{method}1", f"{method}_1", f"{method.upper()}1", f"{method.upper()}_1"]
    possible_y = [f"{method}2", f"{method}_2", f"{method.upper()}2", f"{method.upper()}_2"]
    xcol = next((c for c in possible_x if c in emb.columns), None)
    ycol = next((c for c in possible_y if c in emb.columns), None)

    if xcol is None or ycol is None:
        # If this is a combined embedding table and requested method is missing, fail clearly.
        print("Embedding columns available:", list(emb.columns))
        raise RuntimeError(f"Could not find {method} coordinate columns in selected_raphe embeddings.")

    merged = emb.merge(a[["bodyId", "cluster", "cluster_name", "cluster_size"]], on="bodyId", how="left")
    merged = merged.dropna(subset=["cluster"])
    merged["cluster"] = merged["cluster"].astype(int)
    return ds, merged, xcol, ycol


def check_cells_in_selected_raphe_clusters(cell_ids=None):
    """
    Report which selected-raphe cluster each input cell belongs to.
    This ignores any Raphe-superior-touching dataset.
    """
    if cell_ids is None:
        cell_ids = CELLS_TO_CHECK
    cell_ids = [int(x) for x in cell_ids]

    ds, a = selected_raphe_assignments()
    rows = []

    for bid in cell_ids:
        hit = a.loc[a["bodyId"] == int(bid)]
        if len(hit):
            for _, r in hit.iterrows():
                cl = int(r["cluster"])
                cluster_ids = (
                    a.loc[a["cluster"] == cl, "bodyId"]
                    .astype(int)
                    .sort_values()
                    .tolist()
                )
                rows.append({
                    "bodyId": int(bid),
                    "dataset": SELECTED_RAPHE_DATASET_KEY,
                    "dataset_label": ds.get("label", "Selected raphe neurons"),
                    "cluster": cl,
                    "cluster_name": f"selected_raphe_C{cl:02d}",
                    "cluster_size": len(cluster_ids),
                    "cluster_bodyIds_semicolon": ";".join(map(str, cluster_ids)),
                    "status": "clustered_in_selected_raphe",
                })
        else:
            rows.append({
                "bodyId": int(bid),
                "dataset": SELECTED_RAPHE_DATASET_KEY,
                "dataset_label": ds.get("label", "Selected raphe neurons"),
                "cluster": np.nan,
                "cluster_name": "",
                "cluster_size": np.nan,
                "cluster_bodyIds_semicolon": "",
                "status": "not_found_in_selected_raphe_QC_passing_clusters",
            })

    out = pd.DataFrame(rows)
    display(out)

    path = CHECKED_CELL_FIG_DIR / "checked_cell_cluster_membership_selected_raphe_only.csv"
    out.to_csv(path, index=False)
    print(f"Saved: {path}")
    return out


def plot_checked_cells_on_selected_raphe_embedding(cell_ids=None, method="umap"):
    """
    Plot the original selected-raphe UMAP/MDS colored by cluster and overlay
    the requested IDs as large yellow points/rings.
    """
    if cell_ids is None:
        cell_ids = CELLS_TO_CHECK
    cell_ids = [int(x) for x in cell_ids]
    unique_ids = list(dict.fromkeys(cell_ids))

    ds, emb, xcol, ycol = get_selected_raphe_embedding(method=method)
    present = emb.loc[emb["bodyId"].isin(unique_ids)].copy()
    missing = [bid for bid in unique_ids if bid not in set(present["bodyId"].astype(int))]

    print(f"{method.upper()} overlay for selected_raphe")
    print(f"  requested unique IDs: {len(unique_ids)}")
    print(f"  found in selected-raphe QC/embedding: {len(present)}")
    print(f"  missing from selected-raphe QC/embedding: {len(missing)}")
    if missing:
        print("  missing IDs:", missing)

    fig, ax = plt.subplots(figsize=(6.2, 5.4), dpi=450)

    for cl, sub in emb.groupby("cluster", sort=True):
        ax.scatter(
            sub[xcol], sub[ycol],
            s=CHECK_UMAP_POINT_SIZE,
            alpha=0.85,
            c=get_cluster_color(SELECTED_RAPHE_DATASET_KEY, int(cl)),
            edgecolors="white",
            linewidths=0.35,
            label=f"C{int(cl):02d} (n={len(sub)})",
        )

    if len(present):
        ax.scatter(
            present[xcol], present[ycol],
            s=CHECK_UMAP_HIGHLIGHT_SIZE,
            c=CHECK_UMAP_HIGHLIGHT_COLOR,
            edgecolors=CHECK_UMAP_HIGHLIGHT_EDGE_COLOR,
            linewidths=1.6,
            marker="*",
            label="checked IDs",
            zorder=10,
        )
        if CHECK_UMAP_LABEL_IDS:
            for _, r in present.iterrows():
                ax.text(
                    r[xcol], r[ycol], str(int(r["bodyId"])),
                    fontsize=5.5,
                    weight="bold",
                    ha="left",
                    va="bottom",
                    zorder=11,
                )

    ax.set_title(f"Selected raphe {method.upper()} — checked IDs overlaid")
    ax.set_xlabel(f"{method.upper()} 1")
    ax.set_ylabel(f"{method.upper()} 2")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(False)

    # Keep legend manageable.
    if emb["cluster"].nunique() <= 18:
        ax.legend(frameon=False, fontsize=6, loc="center left", bbox_to_anchor=(1.02, 0.5))

    out_base = CHECKED_CELL_FIG_DIR / f"selected_raphe_{method}_checked_ids_overlay"
    if SAVE_PNG:
        fig.savefig(out_base.with_suffix(".png"), dpi=450, bbox_inches="tight")
    if SAVE_PDF:
        fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    if SAVE_SVG:
        fig.savefig(out_base.with_suffix(".svg"), bbox_inches="tight")
    print(f"Saved checked-ID {method.upper()} overlay to: {out_base}.[png/pdf/svg]")

    display(fig)
    plt.close(fig)

    present_path = CHECKED_CELL_FIG_DIR / f"selected_raphe_{method}_checked_ids_present.csv"
    missing_path = CHECKED_CELL_FIG_DIR / f"selected_raphe_{method}_checked_ids_missing.csv"
    present.to_csv(present_path, index=False)
    pd.DataFrame({"bodyId": missing}).to_csv(missing_path, index=False)
    print("Saved present IDs:", present_path)
    print("Saved missing IDs:", missing_path)
    return fig, present, missing


checked_cells_selected_raphe = check_cells_in_selected_raphe_clusters()
checked_cells_umap_fig, checked_cells_umap_present, checked_cells_umap_missing = plot_checked_cells_on_selected_raphe_embedding(method="umap")
# Optional MDS overlay too:
# checked_cells_mds_fig, checked_cells_mds_present, checked_cells_mds_missing = plot_checked_cells_on_selected_raphe_embedding(method="mds")


## 11. Figure-ready exports — fixed 3D shell views

This section replaces the older 2D projection panels. The old panels used a projected/convex-hull brain outline, which looked ugly and did **not** match the nice interactive 3D brain.

The functions below reuse the same Plotly 3D rendering machinery as the interactive cluster viewer, including the same translucent outer brain shell mesh. The only difference is that the camera is fixed to top, side, and angled views and each view is exported as PDF/PNG/SVG.


In [ ]:

# ============================================================
# 11. Figure-ready export configuration — SELECTED RAPHE ONLY
# ============================================================

FIGURE_READY_DIR = PLOT_OUT_ROOT / "figure_ready_v22_reclusterK_selected_raphe_fixed_3d_shell"
FIGURE_READY_DIR.mkdir(parents=True, exist_ok=True)

FIG_UMAP_DIR = FIGURE_READY_DIR / "umap_mds"
FIG_CLUSTER_3D_DIR = FIGURE_READY_DIR / "cluster_3d_static_views"
FIG_SINGLE_CLUSTER_DIR = FIGURE_READY_DIR / "single_cluster_custom_3d"
for _p in [FIG_UMAP_DIR, FIG_CLUSTER_3D_DIR, FIG_SINGLE_CLUSTER_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

# This notebook/version is only for the selected ~234 raphe neurons.
FIGURE_DATASET_KEY = "selected_raphe"

# 3D static camera views.
FIGURE_3D_VIEWS = ["top", "side", "angled"]
FIGURE_3D_VIEW_LABELS = {
    "top": "Top view",
    "side": "Side view",
    "angled": "Angled view",
}

# Render mode options: "skeletons_only", "somas_only", "combined".
FIGURE_3D_RENDER_MODE = "combined"

# Plotly static output size. Bigger width/height gives nicer PDF/PNG.
FIGURE_3D_WIDTH = 1450
FIGURE_3D_HEIGHT = 1150
FIGURE_3D_SCALE = 2

# These control the same interactive 3D renderer; tune for less crowded figures.
FIGURE_3D_SHELL_ALPHA = 0.060
FIGURE_3D_CELL_LINE_WIDTH = 2.6
FIGURE_3D_CONTEXT_LINE_WIDTH = 0.45
FIGURE_3D_SHOW_CONTEXT = False
FIGURE_3D_SHOW_SOMA_TEXT = False
FIGURE_3D_MAX_CELLS = 120

# Batch rendering controls.
FIGURE_SELECTED_RENDER_ALL_CLUSTERS = True
FIGURE_SELECTED_MIN_CLUSTER_SIZE_TO_RENDER = 2

# UMAP style.
FIGURE_UMAP_POINT_SIZE = 34
FIGURE_UMAP_ALPHA = 0.92
FIGURE_UMAP_EDGE_LW = 0.3
FIGURE_UMAP_LABEL_CLUSTERS = True
FIGURE_UMAP_LABEL_POINTS = False

# Custom cluster defaults.
CUSTOM_DATASET_KEY = FIGURE_DATASET_KEY
CUSTOM_CLUSTER_ID = 1
CUSTOM_3D_VIEWS = ["top", "side", "angled"]
CUSTOM_RENDER_MODE_3D = "combined"
CUSTOM_SHOW_CONTEXT = False
CUSTOM_SHOW_SOMA_TEXT_IN_3D = False
CUSTOM_CELL_LINE_WIDTH = 3.0
CUSTOM_CONTEXT_LINE_WIDTH = 0.45
CUSTOM_SHELL_ALPHA = 0.060
CUSTOM_MAX_CELLS = 120

print("Fixed 3D figure-ready outputs will be saved to:", FIGURE_READY_DIR)
print("Using dataset:", FIGURE_DATASET_KEY)
print("Available datasets:", list(datasets.keys()))


In [ ]:

# ============================================================
# 12. Figure-ready helper functions — UMAPs + fixed-camera 3D exports
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pathlib import Path
from IPython.display import display


def cluster_body_ids(ds, cluster):
    """Return sorted body IDs in a cluster from a loaded dataset dict."""
    if "assignments" not in ds:
        raise KeyError("Dataset dict has no 'assignments' table.")
    a = ds["assignments"].copy()
    a["bodyId"] = a["bodyId"].astype(int)
    a["cluster"] = a["cluster"].astype(int)
    ids = a.loc[a["cluster"] == int(cluster), "bodyId"].astype(int).sort_values().tolist()
    return ids


def _embedding_dataframe_for_method(ds, method):
    """Robustly extract UMAP/MDS coordinates from ds['embeddings'] whether it is a DataFrame or dict."""
    if "embeddings" not in ds:
        return None, None, None
    emb_obj = ds["embeddings"]
    if isinstance(emb_obj, dict):
        emb = emb_obj.get(method)
        if emb is None:
            return None, None, None
        emb = emb.copy()
    else:
        emb = emb_obj.copy()
    if "bodyId" not in emb.columns:
        return None, None, None
    emb["bodyId"] = emb["bodyId"].astype(int)
    possible_x = [f"{method}1", f"{method}_1", f"{method.upper()}1", f"{method.upper()}_1"]
    possible_y = [f"{method}2", f"{method}_2", f"{method.upper()}2", f"{method.upper()}_2"]
    xcol = next((c for c in possible_x if c in emb.columns), None)
    ycol = next((c for c in possible_y if c in emb.columns), None)
    if xcol is None or ycol is None:
        return None, None, None
    return emb, xcol, ycol


def _require_fixed_3d_figure_objects():
    required = [
        "datasets",
        "render_cluster_3d_mode",
        "cluster_body_ids",
        "get_cluster_color",
        "outer_shell_vertices_render",
        "outer_shell_faces",
    ]
    missing = [x for x in required if x not in globals()]
    if missing:
        raise RuntimeError(
            "Missing required variables/functions for fixed 3D exports: "
            + ", ".join(missing)
            + ". Run the earlier loading/rendering cells first."
        )
    if not HAS_PLOTLY:
        raise RuntimeError("Plotly is required for the fixed 3D figure exports.")


def save_matplotlib_all_formats(fig, out_base, dpi=450):
    out_base = Path(out_base)
    out_base.parent.mkdir(parents=True, exist_ok=True)
    paths = []
    if SAVE_PNG:
        p = out_base.with_suffix(".png")
        fig.savefig(p, dpi=dpi, bbox_inches="tight")
        paths.append(p)
    if SAVE_PDF:
        p = out_base.with_suffix(".pdf")
        fig.savefig(p, bbox_inches="tight")
        paths.append(p)
    if SAVE_SVG:
        p = out_base.with_suffix(".svg")
        fig.savefig(p, bbox_inches="tight")
        paths.append(p)
    for p in paths:
        print("Saved:", p)
    return paths


def save_plotly_all_formats(fig, out_base, scale=FIGURE_3D_SCALE, save_html=True):
    """Save a Plotly figure as HTML plus static PNG/PDF/SVG if enabled."""
    out_base = Path(out_base)
    out_base.parent.mkdir(parents=True, exist_ok=True)
    paths = []

    if save_html and SAVE_INTERACTIVE_HTML:
        p = out_base.with_suffix(".html")
        fig.write_html(str(p))
        paths.append(p)
        print("Saved HTML:", p)

    try:
        if SAVE_PNG:
            p = out_base.with_suffix(".png")
            fig.write_image(str(p), scale=scale)
            paths.append(p)
            print("Saved PNG:", p)
        if SAVE_PDF:
            p = out_base.with_suffix(".pdf")
            fig.write_image(str(p), scale=scale)
            paths.append(p)
            print("Saved PDF:", p)
        if SAVE_SVG:
            p = out_base.with_suffix(".svg")
            fig.write_image(str(p), scale=scale)
            paths.append(p)
            print("Saved SVG:", p)
    except Exception as e:
        print(
            "Static Plotly export failed. The HTML/displayed figure is still usable.\n"
            "Install/update kaleido in this environment with:\n"
            "  pip install -U kaleido\n"
            f"Error: {repr(e)}"
        )
    return paths


def fixed_3d_camera(view):
    """Camera presets for static Plotly 3D exports."""
    view = str(view).lower()
    if view == "top":
        return dict(eye=dict(x=0.0, y=0.0, z=2.35), up=dict(x=0, y=1, z=0))
    if view == "side":
        return dict(eye=dict(x=2.35, y=0.0, z=0.0), up=dict(x=0, y=0, z=1))
    if view == "front":
        return dict(eye=dict(x=0.0, y=2.35, z=0.0), up=dict(x=0, y=0, z=1))
    if view == "angled":
        return dict(eye=dict(x=1.55, y=1.65, z=1.15), up=dict(x=0, y=0, z=1))
    raise ValueError(f"Unknown fixed 3D view: {view}")


def polish_fixed_3d_figure(fig, title, view):
    """Make the same interactive 3D brain render look like a clean static figure."""
    fig.update_layout(
        title=dict(text=title, x=0.5, xanchor="center", font=dict(size=22)),
        scene=dict(
            xaxis=dict(visible=False, showbackground=False),
            yaxis=dict(visible=False, showbackground=False),
            zaxis=dict(visible=False, showbackground=False),
            aspectmode="data",
            camera=fixed_3d_camera(view),
            bgcolor="white",
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=0, r=0, t=70, b=0),
        width=FIGURE_3D_WIDTH,
        height=FIGURE_3D_HEIGHT,
        showlegend=False,
    )
    return fig


def _temporarily_set_render_globals(cell_line_width, context_line_width, shell_alpha, max_cells):
    """Context-manager-like helper for temporarily tuning the existing 3D renderer."""
    old = {}
    replacements = {
        "CELL_LINE_WIDTH": cell_line_width,
        "CONTEXT_LINE_WIDTH": context_line_width,
        "SHELL_ALPHA": shell_alpha,
        "MAX_CELLS_PER_CLUSTER_RENDER": max_cells,
    }
    for k, v in replacements.items():
        if k in globals() and v is not None:
            old[k] = globals()[k]
            globals()[k] = v
    return old


def _restore_render_globals(old):
    for k, v in old.items():
        globals()[k] = v


def make_fixed_3d_cluster_view(
    dataset_key=FIGURE_DATASET_KEY,
    cluster=1,
    view="angled",
    render_mode=FIGURE_3D_RENDER_MODE,
    show_context=FIGURE_3D_SHOW_CONTEXT,
    show_soma_text=FIGURE_3D_SHOW_SOMA_TEXT,
    cell_line_width=FIGURE_3D_CELL_LINE_WIDTH,
    context_line_width=FIGURE_3D_CONTEXT_LINE_WIDTH,
    shell_alpha=FIGURE_3D_SHELL_ALPHA,
    max_cells=FIGURE_3D_MAX_CELLS,
    out_dir=None,
    save_outputs=True,
    display_figure=True,
):
    """
    Make one fixed-camera 3D cluster figure using the SAME Plotly brain shell
    as the interactive renderer. This is the key fix relative to the old ugly
    2D shell projection panels.
    """
    _require_fixed_3d_figure_objects()
    if dataset_key not in datasets:
        raise ValueError(f"Unknown dataset_key={dataset_key!r}. Available: {list(datasets.keys())}")

    ds = datasets[dataset_key]
    cluster = int(cluster)
    ids = cluster_body_ids(ds, cluster)
    if len(ids) == 0:
        raise ValueError(f"No cells found for {dataset_key} cluster {cluster}")

    old = _temporarily_set_render_globals(cell_line_width, context_line_width, shell_alpha, max_cells)
    try:
        fig = render_cluster_3d_mode(
            ds,
            dataset_key,
            cluster,
            render_mode=render_mode,
            save_html=False,
            show_population_context=show_context,
            show_soma_text=show_soma_text,
        )
    finally:
        _restore_render_globals(old)

    id_preview = ", ".join(map(str, ids[:12])) + (", ..." if len(ids) > 12 else "")
    title = f"{ds['label']} | C{cluster:02d} | n={len(ids)} | {FIGURE_3D_VIEW_LABELS.get(view, view)}<br><sup>{id_preview}</sup>"
    fig = polish_fixed_3d_figure(fig, title=title, view=view)

    if display_figure:
        display(fig)

    if save_outputs:
        if out_dir is None:
            out_dir = FIG_CLUSTER_3D_DIR / dataset_key / f"C{cluster:02d}"
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        out_base = out_dir / f"{dataset_key}_C{cluster:02d}_{render_mode}_{view}_fixed3D"
        save_plotly_all_formats(fig, out_base, scale=FIGURE_3D_SCALE, save_html=True)

    return fig


def make_fixed_3d_cluster_views(
    dataset_key=FIGURE_DATASET_KEY,
    cluster=1,
    views=None,
    render_mode=FIGURE_3D_RENDER_MODE,
    show_context=FIGURE_3D_SHOW_CONTEXT,
    show_soma_text=FIGURE_3D_SHOW_SOMA_TEXT,
    cell_line_width=FIGURE_3D_CELL_LINE_WIDTH,
    context_line_width=FIGURE_3D_CONTEXT_LINE_WIDTH,
    shell_alpha=FIGURE_3D_SHELL_ALPHA,
    max_cells=FIGURE_3D_MAX_CELLS,
    out_dir=None,
    save_outputs=True,
    display_figures=True,
):
    """Make top/side/angled fixed-camera 3D figures and save each as PDF/PNG/SVG/HTML."""
    if views is None:
        views = FIGURE_3D_VIEWS

    ds = datasets[dataset_key]
    cluster = int(cluster)
    ids = cluster_body_ids(ds, cluster)

    if out_dir is None:
        out_dir = FIG_CLUSTER_3D_DIR / dataset_key / f"C{cluster:02d}"
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    id_table = pd.DataFrame({
        "dataset_key": dataset_key,
        "dataset_label": ds["label"],
        "cluster": cluster,
        "cluster_name": f"{dataset_key}_C{cluster:02d}",
        "n_cells": len(ids),
        "bodyId": ids,
    })
    id_csv = out_dir / f"{dataset_key}_C{cluster:02d}_cell_ids.csv"
    id_txt = out_dir / f"{dataset_key}_C{cluster:02d}_cell_ids.txt"
    id_table.to_csv(id_csv, index=False)
    id_txt.write_text("\n".join(map(str, ids)) + "\n")
    print("Saved cell ID table:", id_csv)
    print("Saved cell ID text:", id_txt)

    figs = {}
    for view in views:
        print("=" * 80)
        print(f"Rendering fixed 3D {view} view | {dataset_key} C{cluster:02d}")
        print("=" * 80)
        figs[view] = make_fixed_3d_cluster_view(
            dataset_key=dataset_key,
            cluster=cluster,
            view=view,
            render_mode=render_mode,
            show_context=show_context,
            show_soma_text=show_soma_text,
            cell_line_width=cell_line_width,
            context_line_width=context_line_width,
            shell_alpha=shell_alpha,
            max_cells=max_cells,
            out_dir=out_dir,
            save_outputs=save_outputs,
            display_figure=display_figures,
        )

    display(id_table)
    return figs, id_table


def plot_embedding_figure_ready(dataset_key=FIGURE_DATASET_KEY, method="umap", label_clusters=FIGURE_UMAP_LABEL_CLUSTERS, label_points=FIGURE_UMAP_LABEL_POINTS):
    """Clean UMAP/MDS plot with stable cluster colors. Robust to DataFrame or dict embedding storage."""
    if dataset_key not in datasets:
        print(f"Skipping {dataset_key}: dataset not loaded.")
        return None
    ds = datasets[dataset_key]
    emb, xcol, ycol = _embedding_dataframe_for_method(ds, method)
    if emb is None:
        print(f"Skipping {dataset_key} {method}: embedding columns not available.")
        if "embeddings" in ds and not isinstance(ds["embeddings"], dict):
            print("Available embedding columns:", list(ds["embeddings"].columns))
        return None

    a = ds["assignments"].copy()
    a["bodyId"] = a["bodyId"].astype(int)
    a["cluster"] = a["cluster"].astype(int)
    if "cluster_name" not in a.columns:
        a["cluster_name"] = a["cluster"].map(lambda x: f"{dataset_key}_C{int(x):02d}")
    if "cluster_size" not in a.columns:
        sizes = a["cluster"].value_counts()
        a["cluster_size"] = a["cluster"].map(sizes).astype(int)

    df = emb.merge(a[["bodyId", "cluster", "cluster_name", "cluster_size"]], on="bodyId", how="left")
    df = df.dropna(subset=["cluster"])
    df["cluster"] = df["cluster"].astype(int)

    fig, ax = plt.subplots(figsize=(5.2, 4.45), dpi=450)
    for cl, sub in df.groupby("cluster", sort=True):
        ax.scatter(
            sub[xcol], sub[ycol],
            s=FIGURE_UMAP_POINT_SIZE,
            alpha=FIGURE_UMAP_ALPHA,
            c=get_cluster_color(dataset_key, int(cl)),
            edgecolors="black",
            linewidths=FIGURE_UMAP_EDGE_LW,
            label=f"C{int(cl):02d} (n={len(sub)})",
        )
        if label_clusters:
            cx, cy = sub[xcol].median(), sub[ycol].median()
            ax.text(cx, cy, f"C{int(cl):02d}", fontsize=7, weight="bold", ha="center", va="center")

    if label_points:
        for _, r in df.iterrows():
            ax.text(r[xcol], r[ycol], str(int(r["bodyId"])), fontsize=4.5, alpha=0.75)

    ax.set_title(f"{ds['label']} — {method.upper()} by morphology cluster")
    ax.set_xlabel(f"{method.upper()} 1")
    ax.set_ylabel(f"{method.upper()} 2")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(False)

    if df["cluster"].nunique() <= 18:
        ax.legend(frameon=False, fontsize=6, loc="center left", bbox_to_anchor=(1.02, 0.5))

    out_base = FIG_UMAP_DIR / f"{dataset_key}_{method}_clusters_figure_ready"
    save_matplotlib_all_formats(fig, out_base, dpi=450)
    plt.show()
    return fig, df


In [ ]:

# ============================================================
# 13. Publication-style UMAP/MDS plots — selected raphe only
# ============================================================

selected_umap_result = plot_embedding_figure_ready(FIGURE_DATASET_KEY, "umap")
selected_mds_result = plot_embedding_figure_ready(FIGURE_DATASET_KEY, "mds")


In [ ]:

# ============================================================
# 14. Batch-render selected-raphe clusters as fixed 3D PDF/PNG/SVG views
# ============================================================

def render_selected_raphe_fixed_3d_cluster_views():
    """Render selected-raphe clusters above the configured size threshold."""
    dataset_key = FIGURE_DATASET_KEY
    if dataset_key not in datasets:
        print(f"{dataset_key} dataset not loaded.")
        return []

    ds = datasets[dataset_key]
    sizes = ds["assignments"]["cluster"].value_counts().sort_index()
    clusters = [int(c) for c, n in sizes.items() if int(n) >= FIGURE_SELECTED_MIN_CLUSTER_SIZE_TO_RENDER]
    if not FIGURE_SELECTED_RENDER_ALL_CLUSTERS:
        clusters = clusters[:8]

    print(f"Rendering {len(clusters)} selected-raphe fixed 3D cluster panels:", clusters)
    outputs = []
    for cl in clusters:
        out_dir = FIG_CLUSTER_3D_DIR / dataset_key / f"C{cl:02d}"
        figs, tab = make_fixed_3d_cluster_views(
            dataset_key=dataset_key,
            cluster=cl,
            views=FIGURE_3D_VIEWS,
            render_mode=FIGURE_3D_RENDER_MODE,
            show_context=FIGURE_3D_SHOW_CONTEXT,
            show_soma_text=FIGURE_3D_SHOW_SOMA_TEXT,
            cell_line_width=FIGURE_3D_CELL_LINE_WIDTH,
            context_line_width=FIGURE_3D_CONTEXT_LINE_WIDTH,
            shell_alpha=FIGURE_3D_SHELL_ALPHA,
            max_cells=FIGURE_3D_MAX_CELLS,
            out_dir=out_dir,
            save_outputs=True,
            display_figures=False,
        )
        outputs.append(out_dir)
    return outputs

# Run batch export. Set FIGURE_SELECTED_RENDER_ALL_CLUSTERS=False for a quick test.
selected_fixed_3d_outputs = render_selected_raphe_fixed_3d_cluster_views()


In [ ]:

# ============================================================
# 15. Custom figure-ready fixed 3D plot for one cluster
# ============================================================
# Edit these parameters and rerun this cell.

CUSTOM_DATASET_KEY = FIGURE_DATASET_KEY
CUSTOM_CLUSTER_ID = 1
CUSTOM_3D_VIEWS = ["top", "side", "angled"]
CUSTOM_RENDER_MODE_3D = "combined"  # "skeletons_only", "somas_only", or "combined"
CUSTOM_SHOW_CONTEXT = False
CUSTOM_SHOW_SOMA_TEXT_IN_3D = False
CUSTOM_CELL_LINE_WIDTH = 3.0
CUSTOM_CONTEXT_LINE_WIDTH = 0.45
CUSTOM_SHELL_ALPHA = 0.060
CUSTOM_MAX_CELLS = 120


def make_custom_cluster_fixed_3d_figure_ready_plot(
    dataset_key=CUSTOM_DATASET_KEY,
    cluster=CUSTOM_CLUSTER_ID,
    views=CUSTOM_3D_VIEWS,
    render_mode_3d=CUSTOM_RENDER_MODE_3D,
    show_context=CUSTOM_SHOW_CONTEXT,
    show_soma_text_3d=CUSTOM_SHOW_SOMA_TEXT_IN_3D,
    cell_line_width=CUSTOM_CELL_LINE_WIDTH,
    context_line_width=CUSTOM_CONTEXT_LINE_WIDTH,
    shell_alpha=CUSTOM_SHELL_ALPHA,
    max_cells=CUSTOM_MAX_CELLS,
):
    """
    Create figure-ready fixed-camera 3D PDFs/PNGs/SVGs for one cluster.

    Unlike the old projection panel, this uses the same nice interactive 3D
    outer brain shell, just with fixed cameras for static export.
    """
    if dataset_key not in datasets:
        raise ValueError(f"Unknown dataset_key={dataset_key!r}. Available: {list(datasets.keys())}")

    ds = datasets[dataset_key]
    cluster = int(cluster)
    ids = cluster_body_ids(ds, cluster)
    if len(ids) == 0:
        raise ValueError(f"No cells found for {dataset_key} cluster {cluster}")

    custom_dir = FIG_SINGLE_CLUSTER_DIR / dataset_key / f"C{cluster:02d}"
    custom_dir.mkdir(parents=True, exist_ok=True)

    print(f"{ds['label']} | C{cluster:02d} | n={len(ids)}")
    display(pd.DataFrame({"bodyId": ids}))

    figs, id_table = make_fixed_3d_cluster_views(
        dataset_key=dataset_key,
        cluster=cluster,
        views=views,
        render_mode=render_mode_3d,
        show_context=show_context,
        show_soma_text=show_soma_text_3d,
        cell_line_width=cell_line_width,
        context_line_width=context_line_width,
        shell_alpha=shell_alpha,
        max_cells=max_cells,
        out_dir=custom_dir,
        save_outputs=True,
        display_figures=True,
    )

    return figs, id_table


custom_fixed_3d_figs, custom_fixed_3d_cell_table = make_custom_cluster_fixed_3d_figure_ready_plot()
